# Procurement Analytics — Análise Estratégica de Compras

**Empresa simulada:** Nexa Distribuição Ltda.  
**Período analisado:** janeiro de 2024 a dezembro de 2025  
**Ferramentas:** Python, Pandas, SQLite, SQL e Plotly

## Sobre o projeto

Este projeto demonstra como dados transacionais de compras podem ser transformados em indicadores para apoiar decisões de suprimentos. A análise cobre gastos, desempenho de fornecedores, variações de preços, oportunidades teóricas de negociação e sinais de risco de abastecimento.

### Objetivos

- Consolidar e validar dados de compras, produtos, fornecedores e entregas.
- Aplicar SQL para criar indicadores e views analíticas reutilizáveis.
- Investigar gasto, OTIF, pontualidade e concentração de fornecedores.
- Estimar oportunidades teóricas de negociação e identificar dependências de fornecimento.
- Comunicar os principais resultados por meio de tabelas, gráficos e recomendações.

### Principais resultados

- **Gasto analisado:** R\$ 142,98 milhões.
- **Pedidos válidos:** 7.772, após excluir pedidos cancelados das análises pertinentes.
- **OTIF global:** 71,69%.
- **Pedidos atrasados:** 23,01%.
- **Economia potencial teórica:** R\$ 62,81 milhões, sujeita a validação comercial.
- **Produtos fora da classificação de baixo risco:** 24, associados a 4,66% do gasto total.

> **Nota metodológica:** os dados são simulados para fins de portfólio. A economia potencial é uma estimativa baseada em preços observados, não uma economia garantida. A classificação de risco é uma triagem inicial e não substitui uma avaliação completa da criticidade operacional.

## Roteiro de leitura

1. Preparação do ambiente e validação dos dados
2. Consultas SQL exploratórias
3. Modelagem analítica com views SQL
4. Dashboard executivo com Python e Plotly
5. Análise de desempenho dos fornecedores
6. Análise de oportunidades de economia
7. Análise de risco de abastecimento
8. Consolidação e conclusão executiva

## 1. Preparação do ambiente e validação dos dados

Nesta etapa, o notebook localiza os arquivos CSV, cria/abre o banco SQLite e valida a integridade básica das tabelas.

**Estrutura esperada do projeto:**

```text
procurement-analytics/
├── Procurement_Analytics_Portfolio_GitHub.ipynb
├── README.md
└── dados/
    ├── categorias.csv
    ├── fornecedores.csv
    ├── produtos.csv
    ├── pedidos_compra.csv
    ├── itens_pedido.csv
    └── entregas.csv
```

O notebook prioriza a pasta `Projeto Procurement Analytics` do Google Drive, se ela existir. Caso contrário, usa a pasta atual de execução, adequada quando o repositório inteiro foi baixado ou clonado.


In [ ]:
# Detecta se o notebook está rodando no Google Colab e monta o Drive somente nesse caso.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path

# Prioriza os dados já existentes no Google Drive; se não existirem,
# usa a pasta do ambiente atual (por exemplo, o Colab aberto pelo GitHub).
drive_project_path = Path('/content/drive/MyDrive/Projeto Procurement Analytics')
local_project_path = Path.cwd()

if IN_COLAB and (drive_project_path / 'dados').is_dir():
    project_path = drive_project_path
else:
    project_path = local_project_path

data_path = project_path / 'dados'
data_path.mkdir(parents=True, exist_ok=True)

print('Pasta do projeto:', project_path.resolve())
print('Pasta dos dados:', data_path.resolve())


['dados', 'Procurement_Analytics_SQL.ipynb', 'procurement_analytics.db']

In [ ]:
import pandas as pd
from urllib.request import urlretrieve

arquivos_csv = [
    'categorias.csv',
    'fornecedores.csv',
    'produtos.csv',
    'pedidos_compra.csv',
    'itens_pedido.csv',
    'entregas.csv',
]

# Se algum CSV não estiver na pasta local/Drive, baixa a cópia pública do GitHub.
base_url = 'https://raw.githubusercontent.com/leonardorsa98/procurement-analytics/main/dados'
arquivos_ausentes = [nome for nome in arquivos_csv if not (data_path / nome).is_file()]

for nome in arquivos_ausentes:
    url = f'{base_url}/{nome}'
    destino = data_path / nome
    try:
        urlretrieve(url, destino)
        print(f'Arquivo baixado do GitHub: {nome}')
    except Exception as exc:
        raise RuntimeError(
            f'Não foi possível obter {nome} do GitHub ({url}). '
            'Confira se o repositório é público e se o arquivo está na pasta dados.'
        ) from exc

arquivos_ausentes = [nome for nome in arquivos_csv if not (data_path / nome).is_file()]
if arquivos_ausentes:
    raise FileNotFoundError(
        f'Arquivos ausentes na pasta {data_path.resolve()}: {", ".join(arquivos_ausentes)}'
    )

categorias = pd.read_csv(data_path / 'categorias.csv')
fornecedores = pd.read_csv(data_path / 'fornecedores.csv')
produtos = pd.read_csv(data_path / 'produtos.csv')
pedidos_compra = pd.read_csv(data_path / 'pedidos_compra.csv')
itens_pedido = pd.read_csv(data_path / 'itens_pedido.csv')
entregas = pd.read_csv(data_path / 'entregas.csv')


In [ ]:
print("Categorias:", categorias.shape)
print("Fornecedores:", fornecedores.shape)
print("Produtos:", produtos.shape)
print("Pedidos:", pedidos_compra.shape)
print("Itens:", itens_pedido.shape)
print("Entregas:", entregas.shape)

Categorias: (10, 2)
Fornecedores: (50, 5)
Produtos: (300, 5)
Pedidos: (8000, 5)
Itens: (20022, 5)
Entregas: (10367, 4)


In [ ]:
import sqlite3

db_path = project_path / 'procurement_analytics.db'
conn = sqlite3.connect(str(db_path))

print('Banco criado/aberto em:', db_path.resolve())


Banco criado em: /content/drive/MyDrive/Projeto Procurement Analytics/procurement_analytics.db


In [ ]:
conn.execute("PRAGMA foreign_keys = ON;")

In [ ]:
conn.executescript("""
CREATE TABLE IF NOT EXISTS categorias (
    categoria_id TEXT PRIMARY KEY,
    nome_categoria TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS fornecedores (
    fornecedor_id TEXT PRIMARY KEY,
    nome_fornecedor TEXT NOT NULL,
    cidade TEXT,
    estado TEXT,
    prazo_contratual_dias INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS produtos (
    produto_id TEXT PRIMARY KEY,
    nome_produto TEXT NOT NULL,
    categoria_id TEXT NOT NULL,
    unidade_medida TEXT NOT NULL,
    preco_base REAL NOT NULL,
    FOREIGN KEY (categoria_id)
        REFERENCES categorias(categoria_id)
);

CREATE TABLE IF NOT EXISTS pedidos_compra (
    pedido_id TEXT PRIMARY KEY,
    fornecedor_id TEXT NOT NULL,
    data_pedido TEXT NOT NULL,
    data_prevista_entrega TEXT NOT NULL,
    status_pedido TEXT NOT NULL,
    FOREIGN KEY (fornecedor_id)
        REFERENCES fornecedores(fornecedor_id)
);

CREATE TABLE IF NOT EXISTS itens_pedido (
    item_id TEXT PRIMARY KEY,
    pedido_id TEXT NOT NULL,
    produto_id TEXT NOT NULL,
    quantidade INTEGER NOT NULL,
    preco_unitario REAL NOT NULL,
    FOREIGN KEY (pedido_id)
        REFERENCES pedidos_compra(pedido_id),
    FOREIGN KEY (produto_id)
        REFERENCES produtos(produto_id)
);

CREATE TABLE IF NOT EXISTS entregas (
    entrega_id TEXT PRIMARY KEY,
    pedido_id TEXT NOT NULL,
    data_entrega TEXT NOT NULL,
    quantidade_entregue INTEGER NOT NULL,
    FOREIGN KEY (pedido_id)
        REFERENCES pedidos_compra(pedido_id)
);
""")

print("Tabelas verificadas: criadas se não existiam.")

Tabelas verificadas: criadas se não existiam.


In [ ]:
tabelas = [
    'categorias',
    'fornecedores',
    'produtos',
    'pedidos_compra',
    'itens_pedido',
    'entregas'
]

# Verifica se todas as tabelas já possuem dados
tabelas_preenchidas = {}

for tabela in tabelas:
    quantidade = conn.execute(
        f"SELECT COUNT(*) FROM {tabela}"
    ).fetchone()[0]

    tabelas_preenchidas[tabela] = quantidade

if all(qtd > 0 for qtd in tabelas_preenchidas.values()):
    print("Dados já carregados. Nenhuma carga foi executada.")

else:
    # Evita carga parcial: se alguma tabela já tiver dados,
    # interrompe para não duplicar ou misturar registros.
    if any(qtd > 0 for qtd in tabelas_preenchidas.values()):
        raise ValueError(
            "Carga parcial detectada. Verifique as tabelas antes de continuar."
        )

    categorias.to_sql(
        'categorias', conn, if_exists='append', index=False
    )

    fornecedores.to_sql(
        'fornecedores', conn, if_exists='append', index=False
    )

    produtos.to_sql(
        'produtos', conn, if_exists='append', index=False
    )

    pedidos_compra.to_sql(
        'pedidos_compra', conn, if_exists='append', index=False
    )

    itens_pedido.to_sql(
        'itens_pedido', conn, if_exists='append', index=False
    )

    entregas.to_sql(
        'entregas', conn, if_exists='append', index=False
    )

    print("Dados carregados com sucesso!")

# Exibe a quantidade de registros de cada tabela
for tabela in tabelas:
    quantidade = conn.execute(
        f"SELECT COUNT(*) FROM {tabela}"
    ).fetchone()[0]

    print(f"{tabela}: {quantidade:,} registros")

Dados já carregados. Nenhuma carga foi executada.
categorias: 10 registros
fornecedores: 50 registros
produtos: 300 registros
pedidos_compra: 8,000 registros
itens_pedido: 20,022 registros
entregas: 10,367 registros


In [ ]:
tabelas = [
    'categorias',
    'fornecedores',
    'produtos',
    'pedidos_compra',
    'itens_pedido',
    'entregas'
]

for tabela in tabelas:
    resultado = pd.read_sql_query(
        f"SELECT COUNT(*) AS quantidade FROM {tabela}",
        conn
    )

    print(f"{tabela}: {resultado.iloc[0]['quantidade']}")

categorias: 10
fornecedores: 50
produtos: 300
pedidos_compra: 8000
itens_pedido: 20022
entregas: 10367


In [ ]:
resultado_fk = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()

print("Problemas de integridade:", len(resultado_fk))

Problemas de integridade: 0


## 2. Consultas SQL exploratórias

As consultas abaixo exploram os dados antes da criação das views. Elas servem para investigar gastos, fornecedores, preços, entregas e concentração. As consultas de desempenho financeiro excluem pedidos cancelados quando indicado no SQL.

### 2.1 — Gastos e tendências de compras

### 2.1 — Gastos e análise financeira


#### 2.1.1 — Gasto total de compras válidas


In [ ]:
query = """
SELECT
    ROUND(
        SUM(i.quantidade * i.preco_unitario),
        2
    ) AS gasto_total
FROM itens_pedido i
JOIN pedidos_compra p
    ON i.pedido_id = p.pedido_id
WHERE p.status_pedido <> 'Cancelado';
"""

pd.read_sql_query(query, conn)

,gasto_total
0,1.429797e+08


#### 2.1.2 — Evolução mensal dos gastos com compras


In [ ]:
query = """
SELECT
    strftime('%Y-%m', p.data_pedido) AS mes,
    ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS gasto_total
FROM itens_pedido i
JOIN pedidos_compra p
    ON i.pedido_id = p.pedido_id
WHERE p.status_pedido <> 'Cancelado'
GROUP BY mes
ORDER BY mes;
"""

pd.read_sql_query(query, conn)

,mes,gasto_total
0,2024-01,5950307.33
1,2024-02,5581586.95
2,2024-03,4978936.12
3,2024-04,5007850.81
4,2024-05,4845458.37
5,2024-06,5618426.32
6,2024-07,6176902.78
7,2024-08,6286781.21
8,2024-09,6204650.18
9,2024-10,6381121.28


#### 2.1.3 — Gastos por categoria


In [ ]:
query = """
SELECT
    c.nome_categoria AS categoria,
    SUM(i.quantidade * i.preco_unitario) AS gasto_total
FROM itens_pedido i
JOIN pedidos_compra p
    ON i.pedido_id = p.pedido_id
JOIN produtos pr
    ON i.produto_id = pr.produto_id
JOIN categorias c
    ON pr.categoria_id = c.categoria_id
WHERE p.status_pedido <> 'Cancelado'
GROUP BY c.nome_categoria
ORDER BY gasto_total DESC
"""

pd.read_sql_query(query, conn)

,categoria,gasto_total
0,Matéria-Prima,34315132.39
1,Embalagens,22876766.88
2,Componentes Industriais,18587359.09
3,Produtos Químicos,15727780.71
4,Material de Manutenção,12868177.09
5,Equipamentos e Ferramentas,11438392.97
6,Higiene e Limpeza,7148993.09
7,Material de Escritório,7148988.12
8,EPIs,7148979.45
9,Outros Suprimentos,5719172.67


#### 2.1.4 — Gastos por fornecedor


In [ ]:
query = """
SELECT
    f.nome_fornecedor AS fornecedor,
    SUM(i.quantidade * i.preco_unitario) AS gasto_total
FROM itens_pedido i
JOIN pedidos_compra p
    ON i.pedido_id = p.pedido_id
JOIN fornecedores f
    ON p.fornecedor_id = f.fornecedor_id
WHERE p.status_pedido <> 'Cancelado'
GROUP BY fornecedor
ORDER BY gasto_total DESC
"""

pd.read_sql_query(query, conn)

,fornecedor,gasto_total
0,Fornecedor 04,11088800.96
1,Fornecedor 05,10521630.42
2,Fornecedor 10,9562174.88
3,Fornecedor 02,8134214.60
4,Fornecedor 09,7161608.40
5,Fornecedor 50,5996246.36
6,Fornecedor 23,5953974.18
7,Fornecedor 17,5679982.84
8,Fornecedor 01,5515668.37
9,Fornecedor 03,5450087.12


#### 2.1.5 — Evolução mensal do gasto total


In [ ]:
query = """
SELECT
    strftime('%Y-%m', p.data_pedido) AS mes,
    ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS gasto_total
FROM itens_pedido i
JOIN pedidos_compra p
    ON i.pedido_id = p.pedido_id
WHERE p.status_pedido <> 'Cancelado'
GROUP BY strftime('%Y-%m', p.data_pedido)
ORDER BY mes ASC
"""

pd.read_sql_query(query, conn)

,mes,gasto_total
0,2024-01,5950307.33
1,2024-02,5581586.95
2,2024-03,4978936.12
3,2024-04,5007850.81
4,2024-05,4845458.37
5,2024-06,5618426.32
6,2024-07,6176902.78
7,2024-08,6286781.21
8,2024-09,6204650.18
9,2024-10,6381121.28


#### 2.1.6 — Ranking de fornecedores por gasto


In [ ]:
query = """
SELECT f.nome_fornecedor, ROUND (SUM (i.quantidade * i.preco_unitario), 2) AS gasto_total
FROM itens_pedido i
JOIN pedidos_compra p
ON i.pedido_id = p.pedido_id
JOIN fornecedores f
ON p.fornecedor_id = f.fornecedor_id
WHERE p.status_pedido <> 'Cancelado'
GROUP BY f.nome_fornecedor
ORDER BY gasto_total DESC
LIMIT 10 """

pd.read_sql_query(query, conn)

,nome_fornecedor,gasto_total
0,Fornecedor 04,11088800.96
1,Fornecedor 05,10521630.42
2,Fornecedor 10,9562174.88
3,Fornecedor 02,8134214.60
4,Fornecedor 09,7161608.40
5,Fornecedor 50,5996246.36
6,Fornecedor 23,5953974.18
7,Fornecedor 17,5679982.84
8,Fornecedor 01,5515668.37
9,Fornecedor 03,5450087.12


#### 2.1.7 — Concentração acumulada dos gastos por fornecedor


In [ ]:
query = """
SELECT
    f.nome_fornecedor,

    ROUND(
        SUM(i.quantidade * i.preco_unitario),
        2
    ) AS gasto_total,

    ROUND(
        SUM(i.quantidade * i.preco_unitario)
        /
        (
            SELECT
                SUM(i.quantidade * i.preco_unitario)
            FROM itens_pedido i
            JOIN pedidos_compra p
                ON i.pedido_id = p.pedido_id
            WHERE p.status_pedido <> 'Cancelado'
        )
        * 100,
        2
    ) AS percentual_total

FROM itens_pedido i

JOIN pedidos_compra p
    ON i.pedido_id = p.pedido_id

JOIN fornecedores f
    ON p.fornecedor_id = f.fornecedor_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY f.nome_fornecedor

ORDER BY gasto_total DESC;
"""

pd.read_sql_query(query, conn)


,nome_fornecedor,gasto_total,percentual_total
0,Fornecedor 04,11088800.96,7.76
1,Fornecedor 05,10521630.42,7.36
2,Fornecedor 10,9562174.88,6.69
3,Fornecedor 02,8134214.60,5.69
4,Fornecedor 09,7161608.40,5.01
5,Fornecedor 50,5996246.36,4.19
6,Fornecedor 23,5953974.18,4.16
7,Fornecedor 17,5679982.84,3.97
8,Fornecedor 01,5515668.37,3.86
9,Fornecedor 03,5450087.12,3.81


#### 2.1.8 — Volume comprado e gasto por produto


In [ ]:
query = """
SELECT
    pr.nome_produto AS produto,

    SUM(i.quantidade) AS quantidade_total,

    ROUND(
        SUM(i.quantidade * i.preco_unitario),
        2
    ) AS gasto_total

FROM itens_pedido i

JOIN pedidos_compra p
    ON i.pedido_id = p.pedido_id

JOIN produtos pr
    ON i.produto_id = pr.produto_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY pr.nome_produto

ORDER BY gasto_total DESC

LIMIT 20;
"""

pd.read_sql_query(query, conn)

,produto,quantidade_total,gasto_total
0,Produto 178,13900,7476785.69
1,Produto 126,17603,6344395.72
2,Produto 086,16883,5487712.46
3,Produto 054,26734,4443137.52
4,Produto 242,10704,4368701.41
5,Produto 291,13677,4173884.83
6,Produto 272,9043,3893212.62
7,Produto 045,23193,3660808.33
8,Produto 245,5322,3601450.30
9,Produto 205,14811,3420156.49


### 2.2 — Desempenho de fornecedores e entregas

#### 2.2.1 — Prazo médio de entrega por fornecedor


In [ ]:
query = """
SELECT
    f.nome_fornecedor AS fornecedor,

    ROUND(
        AVG(
            julianday(e.data_entrega) - julianday(p.data_pedido)
        ),
        2
    ) AS prazo_medio_entrega_dias

FROM pedidos_compra p

JOIN entregas e
    ON p.pedido_id = e.pedido_id

JOIN fornecedores f
    ON p.fornecedor_id = f.fornecedor_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY f.nome_fornecedor

ORDER BY prazo_medio_entrega_dias ASC;
"""

pd.read_sql_query(query, conn)

,fornecedor,prazo_medio_entrega_dias
0,Fornecedor 48,3.08
1,Fornecedor 12,3.24
2,Fornecedor 34,3.47
3,Fornecedor 19,3.56
4,Fornecedor 41,3.76
5,Fornecedor 11,3.77
6,Fornecedor 22,3.81
7,Fornecedor 16,4.09
8,Fornecedor 33,4.14
9,Fornecedor 23,4.99


#### 2.2.2 — Indicadores de entregas atrasadas por fornecedor


In [ ]:
query = """
SELECT
    f.nome_fornecedor AS fornecedor,

    COUNT(e.entrega_id) AS total_entregas,

    SUM(
        CASE
            WHEN e.data_entrega > p.data_prevista_entrega THEN 1
            ELSE 0
        END
    ) AS entregas_atrasadas,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN e.data_entrega > p.data_prevista_entrega THEN 1
                ELSE 0
            END
        ) / COUNT(e.entrega_id),
        2
    ) AS percentual_atraso

FROM pedidos_compra p

JOIN entregas e
    ON p.pedido_id = e.pedido_id

JOIN fornecedores f
    ON p.fornecedor_id = f.fornecedor_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY f.nome_fornecedor

ORDER BY percentual_atraso DESC;
"""

pd.read_sql_query(query, conn)

,fornecedor,total_entregas,entregas_atrasadas,percentual_atraso
0,Fornecedor 09,395,131,33.16
1,Fornecedor 08,364,113,31.04
2,Fornecedor 29,123,38,30.89
3,Fornecedor 02,542,161,29.70
4,Fornecedor 05,484,142,29.34
5,Fornecedor 03,335,97,28.96
6,Fornecedor 20,112,32,28.57
7,Fornecedor 01,492,139,28.25
8,Fornecedor 32,132,36,27.27
9,Fornecedor 13,111,30,27.03


#### 2.2.3 — Comparação entre prazo contratual e prazo real


In [ ]:
query = """
SELECT
    f.nome_fornecedor AS fornecedor,

    f.prazo_contratual_dias AS prazo_contratual_dias,

    ROUND(
        AVG(
            julianday(e.data_entrega) - julianday(p.data_pedido)
        ),
        2
    ) AS prazo_medio_real_dias,

    ROUND(
        AVG(
            julianday(e.data_entrega) - julianday(p.data_pedido)
        ) - f.prazo_contratual_dias,
        2
    ) AS diferenca_media_dias

FROM pedidos_compra p

JOIN entregas e
    ON p.pedido_id = e.pedido_id

JOIN fornecedores f
    ON p.fornecedor_id = f.fornecedor_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY
    f.nome_fornecedor,
    f.prazo_contratual_dias

ORDER BY diferenca_media_dias DESC;
"""

pd.read_sql_query(query, conn)

,fornecedor,prazo_contratual_dias,prazo_medio_real_dias,diferenca_media_dias
0,Fornecedor 33,3,4.14,1.14
1,Fornecedor 08,14,14.90,0.90
2,Fornecedor 09,9,9.86,0.86
3,Fornecedor 22,3,3.81,0.81
4,Fornecedor 18,13,13.68,0.68
5,Fornecedor 20,14,14.61,0.61
6,Fornecedor 19,3,3.56,0.56
7,Fornecedor 29,14,14.56,0.56
8,Fornecedor 35,13,13.55,0.55
9,Fornecedor 34,3,3.47,0.47


#### 2.2.4 — Entregas atrasadas por fornecedor


In [ ]:
query = """
SELECT
    f.nome_fornecedor AS fornecedor,

    COUNT(e.entrega_id) AS total_entregas,

    SUM(
        CASE
            WHEN e.data_entrega > p.data_prevista_entrega THEN 1
            ELSE 0
        END
    ) AS entregas_atrasadas,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN e.data_entrega > p.data_prevista_entrega THEN 1
                ELSE 0
            END
        ) / COUNT(e.entrega_id),
        2
    ) AS percentual_atraso

FROM pedidos_compra p

JOIN entregas e
    ON p.pedido_id = e.pedido_id

JOIN fornecedores f
    ON p.fornecedor_id = f.fornecedor_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY f.nome_fornecedor

HAVING COUNT(e.entrega_id) >= 50

ORDER BY percentual_atraso DESC;
"""

pd.read_sql_query(query, conn)

,fornecedor,total_entregas,entregas_atrasadas,percentual_atraso
0,Fornecedor 09,395,131,33.16
1,Fornecedor 08,364,113,31.04
2,Fornecedor 29,123,38,30.89
3,Fornecedor 02,542,161,29.70
4,Fornecedor 05,484,142,29.34
5,Fornecedor 03,335,97,28.96
6,Fornecedor 20,112,32,28.57
7,Fornecedor 01,492,139,28.25
8,Fornecedor 32,132,36,27.27
9,Fornecedor 13,111,30,27.03


#### 2.2.5 — Ticket médio por fornecedor


In [ ]:
query = """
SELECT
    f.nome_fornecedor AS fornecedor,

    COUNT(DISTINCT p.pedido_id) AS quantidade_pedidos,

    ROUND(
        SUM(i.quantidade * i.preco_unitario),
        2
    ) AS gasto_total,

    ROUND(
        SUM(i.quantidade * i.preco_unitario)
        / COUNT(DISTINCT p.pedido_id),
        2
    ) AS valor_medio_pedido

FROM pedidos_compra p

JOIN itens_pedido i
    ON p.pedido_id = i.pedido_id

JOIN fornecedores f
    ON p.fornecedor_id = f.fornecedor_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY f.nome_fornecedor

ORDER BY valor_medio_pedido DESC;
"""

pd.read_sql_query(query, conn)

,fornecedor,quantidade_pedidos,gasto_total,valor_medio_pedido
0,Fornecedor 32,100,4454116.23,44541.16
1,Fornecedor 23,141,5953974.18,42226.77
2,Fornecedor 19,88,3237153.05,36785.83
3,Fornecedor 26,83,3042886.49,36661.28
4,Fornecedor 10,305,9562174.88,31351.39
5,Fornecedor 05,361,10521630.42,29145.79
6,Fornecedor 50,206,5996246.36,29107.99
7,Fornecedor 27,110,3094746.47,28134.06
8,Fornecedor 17,223,5679982.84,25470.78
9,Fornecedor 38,126,3179382.79,25233.20


### 2.3 — Preços e variações de compra

#### 2.3.1 — Evolução mensal do preço médio de compra


In [ ]:
query = """
SELECT
    strftime('%Y-%m', p.data_pedido) AS mes,

    ROUND(
        AVG(i.preco_unitario),
        2
    ) AS preco_medio_compra

FROM pedidos_compra p

JOIN itens_pedido i
    ON p.pedido_id = i.pedido_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY strftime('%Y-%m', p.data_pedido)

ORDER BY mes;
"""

pd.read_sql_query(query, conn)

,mes,preco_medio_compra
0,2024-01,151.35
1,2024-02,149.86
2,2024-03,150.42
3,2024-04,149.98
4,2024-05,161.70
5,2024-06,149.94
6,2024-07,169.47
7,2024-08,157.57
8,2024-09,156.67
9,2024-10,159.36


#### 2.3.2 — Variação de preços por produto ao longo do período


In [ ]:
query = """
WITH preco_produto AS (
    SELECT
        i.produto_id,
        pr.nome_produto AS produto,

        AVG(
            CASE
                WHEN p.data_pedido = (
                    SELECT MIN(p2.data_pedido)
                    FROM pedidos_compra p2
                    WHERE p2.status_pedido <> 'Cancelado'
                )
                THEN i.preco_unitario
            END
        ) AS preco_inicial,

        AVG(
            CASE
                WHEN p.data_pedido = (
                    SELECT MAX(p3.data_pedido)
                    FROM pedidos_compra p3
                    WHERE p3.status_pedido <> 'Cancelado'
                )
                THEN i.preco_unitario
            END
        ) AS preco_final

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        i.produto_id,
        pr.nome_produto
)

SELECT
    produto,

    ROUND(preco_inicial, 2) AS preco_inicial,

    ROUND(preco_final, 2) AS preco_final,

    ROUND(
        ((preco_final - preco_inicial) / preco_inicial) * 100,
        2
    ) AS variacao_percentual

FROM preco_produto

WHERE preco_inicial IS NOT NULL
  AND preco_final IS NOT NULL
  AND preco_inicial > 0

ORDER BY variacao_percentual DESC

LIMIT 20;
"""

pd.read_sql_query(query, conn)

,produto,preco_inicial,preco_final,variacao_percentual
0,Produto 186,107.71,199.67,85.38
1,Produto 110,28.93,46.34,60.15
2,Produto 086,375.98,398.46,5.98
3,Produto 209,343.25,334.12,-2.66


#### 2.3.3 — Amplitude de preços por produto


In [ ]:
query = """
SELECT
    pr.nome_produto AS produto,

    MIN(i.preco_unitario) AS menor_preco,

    MAX(i.preco_unitario) AS maior_preco,

    ROUND(
        MAX(i.preco_unitario) - MIN(i.preco_unitario),
        2
    ) AS diferenca_preco,

    ROUND(
        (
            MAX(i.preco_unitario) - MIN(i.preco_unitario)
        )
        / MIN(i.preco_unitario) * 100,
        2
    ) AS diferenca_percentual

FROM itens_pedido i

JOIN pedidos_compra p
    ON i.pedido_id = p.pedido_id

JOIN produtos pr
    ON i.produto_id = pr.produto_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY pr.nome_produto

HAVING COUNT(DISTINCT p.fornecedor_id) >= 2

ORDER BY diferenca_percentual DESC

LIMIT 20;
"""

pd.read_sql_query(query, conn)

,produto,menor_preco,maior_preco,diferenca_preco,diferenca_percentual
0,Produto 190,24.76,86.07,61.31,247.62
1,Produto 045,69.64,239.90,170.26,244.49
2,Produto 291,137.18,471.61,334.43,243.79
3,Produto 105,44.60,153.13,108.53,243.34
4,Produto 178,233.59,801.68,568.09,243.20
5,Produto 220,182.32,607.09,424.77,232.98
6,Produto 186,97.76,323.89,226.13,231.31
7,Produto 048,20.25,66.83,46.58,230.02
8,Produto 107,232.05,761.92,529.87,228.34
9,Produto 075,258.71,849.19,590.48,228.24


#### 2.3.4 — Concentração de gastos por fornecedor


In [ ]:
query = """
WITH gasto_fornecedor AS (

    SELECT
        f.nome_fornecedor AS fornecedor,

        SUM(
            i.quantidade * i.preco_unitario
        ) AS gasto_total

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN fornecedores f
        ON p.fornecedor_id = f.fornecedor_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY f.nome_fornecedor
)

SELECT
    fornecedor,

    ROUND(gasto_total, 2) AS gasto_total,

    ROUND(
        gasto_total
        /
        SUM(gasto_total) OVER ()
        * 100,
        2
    ) AS percentual_gasto,

    ROUND(
        SUM(gasto_total) OVER (
            ORDER BY gasto_total DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
        /
        SUM(gasto_total) OVER ()
        * 100,
        2
    ) AS percentual_acumulado

FROM gasto_fornecedor

ORDER BY gasto_total DESC;
"""

pd.read_sql_query(query, conn)

,fornecedor,gasto_total,percentual_gasto,percentual_acumulado
0,Fornecedor 04,11088800.96,7.76,7.76
1,Fornecedor 05,10521630.42,7.36,15.11
2,Fornecedor 10,9562174.88,6.69,21.80
3,Fornecedor 02,8134214.60,5.69,27.49
4,Fornecedor 09,7161608.40,5.01,32.50
5,Fornecedor 50,5996246.36,4.19,36.69
6,Fornecedor 23,5953974.18,4.16,40.86
7,Fornecedor 17,5679982.84,3.97,44.83
8,Fornecedor 01,5515668.37,3.86,48.69
9,Fornecedor 03,5450087.12,3.81,52.50


#### 2.3.5 — Indicadores financeiros e operacionais dos fornecedores


In [ ]:
query = """
WITH fornecedor_financeiro AS (

    SELECT
        f.fornecedor_id,
        f.nome_fornecedor AS fornecedor,

        ROUND(
            SUM(i.quantidade * i.preco_unitario),
            2
        ) AS gasto_total

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN fornecedores f
        ON p.fornecedor_id = f.fornecedor_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        f.fornecedor_id,
        f.nome_fornecedor
),

fornecedor_operacional AS (

    SELECT
        f.fornecedor_id,

        COUNT(e.entrega_id) AS total_entregas,

        SUM(
            CASE
                WHEN e.data_entrega > p.data_prevista_entrega THEN 1
                ELSE 0
            END
        ) AS entregas_atrasadas,

        ROUND(
            100.0 * SUM(
                CASE
                    WHEN e.data_entrega > p.data_prevista_entrega THEN 1
                    ELSE 0
                END
            ) / COUNT(e.entrega_id),
            2
        ) AS percentual_atraso

    FROM pedidos_compra p

    JOIN entregas e
        ON p.pedido_id = e.pedido_id

    JOIN fornecedores f
        ON p.fornecedor_id = f.fornecedor_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY f.fornecedor_id
)

SELECT
    ff.fornecedor,
    ff.gasto_total,
    fo.total_entregas,
    fo.entregas_atrasadas,
    fo.percentual_atraso

FROM fornecedor_financeiro ff

JOIN fornecedor_operacional fo
    ON ff.fornecedor_id = fo.fornecedor_id

ORDER BY
    ff.gasto_total DESC;
"""

pd.read_sql_query(query, conn)

,fornecedor,gasto_total,total_entregas,entregas_atrasadas,percentual_atraso
0,Fornecedor 04,11088800.96,731,125,17.10
1,Fornecedor 05,10521630.42,484,142,29.34
2,Fornecedor 10,9562174.88,395,87,22.03
3,Fornecedor 02,8134214.60,542,161,29.70
4,Fornecedor 09,7161608.40,395,131,33.16
5,Fornecedor 50,5996246.36,295,56,18.98
6,Fornecedor 23,5953974.18,166,21,12.65
7,Fornecedor 17,5679982.84,279,34,12.19
8,Fornecedor 01,5515668.37,492,139,28.25
9,Fornecedor 03,5450087.12,335,97,28.96


#### 2.3.6 — Resumo de preços e oportunidades por produto


In [ ]:
query = """
WITH resumo_produto AS (

    SELECT
        pr.produto_id,
        pr.nome_produto AS produto,

        SUM(i.quantidade) AS quantidade_total,

        AVG(i.preco_unitario) AS preco_medio,

        MIN(i.preco_unitario) AS menor_preco,

        MAX(i.preco_unitario) AS maior_preco

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        pr.produto_id,
        pr.nome_produto
)

SELECT
    produto,

    quantidade_total,

    ROUND(preco_medio, 2) AS preco_medio,

    ROUND(menor_preco, 2) AS menor_preco,

    ROUND(maior_preco, 2) AS maior_preco,

    ROUND(
        preco_medio - menor_preco,
        2
    ) AS diferenca_media_para_menor,

    ROUND(
        (
            (preco_medio - menor_preco)
            / menor_preco
        ) * 100,
        2
    ) AS potencial_percentual

FROM resumo_produto

WHERE menor_preco > 0

ORDER BY potencial_percentual DESC

LIMIT 20;
"""

pd.read_sql_query(query, conn)

,produto,quantidade_total,preco_medio,menor_preco,maior_preco,diferenca_media_para_menor,potencial_percentual
0,Produto 257,246,69.70,26.85,80.34,42.85,159.59
1,Produto 247,506,551.83,226.11,732.17,325.72,144.05
2,Produto 105,4091,108.16,44.60,153.13,63.56,142.51
3,Produto 252,5887,562.79,245.23,665.66,317.56,129.49
4,Produto 178,13900,533.23,233.59,801.68,299.64,128.28
5,Produto 187,3668,191.70,84.07,228.42,107.63,128.03
6,Produto 156,3457,79.91,35.22,99.77,44.69,126.88
7,Produto 045,23193,156.86,69.64,239.90,87.22,125.25
8,Produto 107,3854,521.38,232.05,761.92,289.33,124.68
9,Produto 193,3746,35.13,15.64,39.46,19.49,124.61


### 2.4 — Concentração e risco de fornecimento

#### 2.4.1 — Dependência de fornecedores por produto


In [ ]:
query = """
SELECT
    pr.nome_produto AS produto,

    COUNT(DISTINCT p.fornecedor_id) AS quantidade_fornecedores,

    COUNT(DISTINCT p.pedido_id) AS quantidade_pedidos,

    SUM(i.quantidade) AS quantidade_total,

    ROUND(
        SUM(i.quantidade * i.preco_unitario),
        2
    ) AS gasto_total

FROM itens_pedido i

JOIN produtos pr
    ON i.produto_id = pr.produto_id

JOIN pedidos_compra p
    ON i.pedido_id = p.pedido_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY pr.nome_produto

ORDER BY quantidade_fornecedores ASC,
         gasto_total DESC

LIMIT 30;
"""

pd.read_sql_query(query, conn)

,produto,quantidade_fornecedores,quantidade_pedidos,quantidade_total,gasto_total
0,Produto 208,1,29,1074,605458.24
1,Produto 100,1,33,1778,244695.00
2,Produto 125,1,14,1014,27671.57
3,Produto 141,1,10,166,15954.22
4,Produto 160,1,11,476,13686.31
5,Produto 113,1,7,117,12322.08
6,Produto 198,1,4,98,11618.01
7,Produto 081,1,7,249,10293.55
8,Produto 114,1,7,250,8729.75
9,Produto 192,1,14,389,4855.44


#### 2.4.2 — Concentração de gastos por categoria e fornecedor


In [ ]:
query = """
WITH gasto_categoria_fornecedor AS (

    SELECT
        c.categoria_id,
        c.nome_categoria AS categoria,
        f.fornecedor_id,
        f.nome_fornecedor AS fornecedor,

        SUM(
            i.quantidade * i.preco_unitario
        ) AS gasto_total

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    JOIN categorias c
        ON pr.categoria_id = c.categoria_id

    JOIN fornecedores f
        ON p.fornecedor_id = f.fornecedor_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        c.categoria_id,
        c.nome_categoria,
        f.fornecedor_id,
        f.nome_fornecedor
)

SELECT
    categoria,
    fornecedor,

    ROUND(
        gasto_total,
        2
    ) AS gasto_total,

    ROUND(
        100.0 * gasto_total
        /
        SUM(gasto_total) OVER (
            PARTITION BY categoria_id
        ),
        2
    ) AS percentual_categoria

FROM gasto_categoria_fornecedor

ORDER BY
    categoria,
    percentual_categoria DESC;
"""

pd.read_sql_query(query, conn)

,categoria,fornecedor,gasto_total,percentual_categoria
0,Componentes Industriais,Fornecedor 09,4266145.93,22.95
1,Componentes Industriais,Fornecedor 04,2285787.14,12.30
2,Componentes Industriais,Fornecedor 05,1692006.54,9.10
3,Componentes Industriais,Fornecedor 32,1180674.50,6.35
4,Componentes Industriais,Fornecedor 02,721129.61,3.88
...,...,...,...,...
426,Produtos Químicos,Fornecedor 39,5751.55,0.04
427,Produtos Químicos,Fornecedor 30,4724.91,0.03
428,Produtos Químicos,Fornecedor 21,2596.70,0.02
429,Produtos Químicos,Fornecedor 16,218.19,0.00


#### 2.4.3 — Participação dos fornecedores dentro de cada categoria


In [ ]:
query = """
WITH gasto_categoria_fornecedor AS (

    SELECT
        c.categoria_id,
        c.nome_categoria AS categoria,
        f.fornecedor_id,
        f.nome_fornecedor AS fornecedor,

        SUM(
            i.quantidade * i.preco_unitario
        ) AS gasto_total

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    JOIN categorias c
        ON pr.categoria_id = c.categoria_id

    JOIN fornecedores f
        ON p.fornecedor_id = f.fornecedor_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        c.categoria_id,
        c.nome_categoria,
        f.fornecedor_id,
        f.nome_fornecedor
)

SELECT
    categoria,
    fornecedor,

    ROUND(
        gasto_total,
        2
    ) AS gasto_total,

    ROUND(
        100.0 * gasto_total
        /
        SUM(gasto_total) OVER (
            PARTITION BY categoria_id
        ),
        2
    ) AS percentual_categoria

FROM gasto_categoria_fornecedor

ORDER BY
    categoria,
    percentual_categoria DESC;
"""

pd.read_sql_query(query, conn)

,categoria,fornecedor,gasto_total,percentual_categoria
0,Componentes Industriais,Fornecedor 09,4266145.93,22.95
1,Componentes Industriais,Fornecedor 04,2285787.14,12.30
2,Componentes Industriais,Fornecedor 05,1692006.54,9.10
3,Componentes Industriais,Fornecedor 32,1180674.50,6.35
4,Componentes Industriais,Fornecedor 02,721129.61,3.88
...,...,...,...,...
426,Produtos Químicos,Fornecedor 39,5751.55,0.04
427,Produtos Químicos,Fornecedor 30,4724.91,0.03
428,Produtos Químicos,Fornecedor 21,2596.70,0.02
429,Produtos Químicos,Fornecedor 16,218.19,0.00


#### 2.4.4 — Comparação de preços entre fornecedores para o mesmo produto


In [ ]:
query = """
WITH preco_medio_produto AS (

    SELECT
        i.produto_id,

        AVG(i.preco_unitario) AS preco_medio

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY i.produto_id
),

preco_fornecedor AS (

    SELECT
        i.produto_id,
        pr.nome_produto AS produto,

        p.fornecedor_id,
        f.nome_fornecedor AS fornecedor,

        AVG(i.preco_unitario) AS preco_medio_fornecedor

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    JOIN fornecedores f
        ON p.fornecedor_id = f.fornecedor_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        i.produto_id,
        pr.nome_produto,
        p.fornecedor_id,
        f.nome_fornecedor
)

SELECT
    pf.produto,
    pf.fornecedor,

    ROUND(
        pf.preco_medio_fornecedor,
        2
    ) AS preco_medio_fornecedor,

    ROUND(
        pmp.preco_medio,
        2
    ) AS preco_medio_produto,

    ROUND(
        pf.preco_medio_fornecedor
        - pmp.preco_medio,
        2
    ) AS diferenca_preco,

    ROUND(
        100.0 *
        (
            pf.preco_medio_fornecedor
            - pmp.preco_medio
        )
        / pmp.preco_medio,
        2
    ) AS diferenca_percentual

FROM preco_fornecedor pf

JOIN preco_medio_produto pmp
    ON pf.produto_id = pmp.produto_id

WHERE pf.preco_medio_fornecedor > pmp.preco_medio

ORDER BY diferenca_percentual DESC

LIMIT 30;
"""

pd.read_sql_query(query, conn)

,produto,fornecedor,preco_medio_fornecedor,preco_medio_produto,diferenca_preco,diferenca_percentual
0,Produto 166,Fornecedor 23,221.68,101.54,120.13,118.31
1,Produto 048,Fornecedor 23,63.96,31.50,32.46,103.06
2,Produto 226,Fornecedor 23,1245.91,629.50,616.41,97.92
3,Produto 132,Fornecedor 23,270.04,143.59,126.44,88.05
4,Produto 144,Fornecedor 19,40.92,21.89,19.03,86.96
5,Produto 255,Fornecedor 26,6.51,3.54,2.97,83.68
6,Produto 006,Fornecedor 19,16.51,9.04,7.47,82.64
7,Produto 109,Fornecedor 23,28.26,15.72,12.55,79.84
8,Produto 204,Fornecedor 26,132.58,74.07,58.51,78.99
9,Produto 186,Fornecedor 23,308.78,173.95,134.83,77.51


#### 2.4.5 — Ranking de fornecedores por participação na categoria


In [ ]:
query = """
WITH gasto_categoria_fornecedor AS (

    SELECT
        c.categoria_id,
        c.nome_categoria AS categoria,

        f.fornecedor_id,
        f.nome_fornecedor AS fornecedor,

        SUM(
            i.quantidade * i.preco_unitario
        ) AS gasto_total

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    JOIN categorias c
        ON pr.categoria_id = c.categoria_id

    JOIN fornecedores f
        ON p.fornecedor_id = f.fornecedor_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        c.categoria_id,
        c.nome_categoria,
        f.fornecedor_id,
        f.nome_fornecedor
)

SELECT
    categoria,
    fornecedor,

    ROUND(
        gasto_total,
        2
    ) AS gasto_total,

    RANK() OVER (
        PARTITION BY categoria_id
        ORDER BY gasto_total DESC
    ) AS ranking_categoria

FROM gasto_categoria_fornecedor

ORDER BY
    categoria,
    ranking_categoria;
"""

pd.read_sql_query(query, conn)


,categoria,fornecedor,gasto_total,ranking_categoria
0,Componentes Industriais,Fornecedor 09,4266145.93,1
1,Componentes Industriais,Fornecedor 04,2285787.14,2
2,Componentes Industriais,Fornecedor 05,1692006.54,3
3,Componentes Industriais,Fornecedor 32,1180674.50,4
4,Componentes Industriais,Fornecedor 02,721129.61,5
...,...,...,...,...
426,Produtos Químicos,Fornecedor 39,5751.55,45
427,Produtos Químicos,Fornecedor 30,4724.91,46
428,Produtos Químicos,Fornecedor 21,2596.70,47
429,Produtos Químicos,Fornecedor 47,496.44,48


### 2.5 — Comparação de preços entre fornecedores

#### 2.5.1 — Comparação de preços entre fornecedores


In [ ]:
query = """
WITH preco_fornecedor AS (

    SELECT
        i.produto_id,
        pr.nome_produto AS produto,

        p.fornecedor_id,
        f.nome_fornecedor AS fornecedor,

        AVG(i.preco_unitario) AS preco_medio

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    JOIN fornecedores f
        ON p.fornecedor_id = f.fornecedor_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        i.produto_id,
        pr.nome_produto,
        p.fornecedor_id,
        f.nome_fornecedor
),

ranking AS (

    SELECT
        produto_id,
        produto,
        fornecedor_id,
        fornecedor,
        preco_medio,

        ROW_NUMBER() OVER (
            PARTITION BY produto_id
            ORDER BY preco_medio ASC
        ) AS ranking_preco

    FROM preco_fornecedor
)

SELECT
    produto,
    fornecedor,

    ROUND(
        preco_medio,
        2
    ) AS preco_medio,

    ranking_preco

FROM ranking

WHERE ranking_preco = 1

ORDER BY preco_medio ASC;
"""

pd.read_sql_query(query, conn)

,produto,fornecedor,preco_medio,ranking_preco
0,Produto 255,Fornecedor 30,3.21,1
1,Produto 128,Fornecedor 22,3.46,1
2,Produto 200,Fornecedor 22,3.50,1
3,Produto 168,Fornecedor 21,3.62,1
4,Produto 235,Fornecedor 14,3.65,1
...,...,...,...,...
294,Produto 251,Fornecedor 46,376.76,1
295,Produto 280,Fornecedor 06,384.30,1
296,Produto 226,Fornecedor 49,422.57,1
297,Produto 208,Fornecedor 04,562.14,1


#### 2.5.2 — Evolução mensal dos preços por produto


In [ ]:
query = """
WITH preco_mensal AS (

    SELECT
        i.produto_id,
        pr.nome_produto AS produto,

        strftime('%Y-%m', p.data_pedido) AS mes,

        AVG(i.preco_unitario) AS preco_medio

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        i.produto_id,
        pr.nome_produto,
        strftime('%Y-%m', p.data_pedido)
),

evolucao AS (

    SELECT
        produto_id,
        produto,
        mes,
        preco_medio,

        LAG(preco_medio) OVER (
            PARTITION BY produto_id
            ORDER BY mes
        ) AS preco_mes_anterior

    FROM preco_mensal
)

SELECT
    produto,
    mes,

    ROUND(
        preco_medio,
        2
    ) AS preco_medio,

    ROUND(
        preco_mes_anterior,
        2
    ) AS preco_mes_anterior,

    ROUND(
        preco_medio - preco_mes_anterior,
        2
    ) AS variacao_valor,

    ROUND(
        100.0 *
        (preco_medio - preco_mes_anterior)
        / preco_mes_anterior,
        2
    ) AS variacao_percentual

FROM evolucao

WHERE preco_mes_anterior IS NOT NULL

ORDER BY
    produto,
    mes;
"""

pd.read_sql_query(query, conn)

,produto,mes,preco_medio,preco_mes_anterior,variacao_valor,variacao_percentual
0,Produto 001,2024-02,24.76,23.24,1.52,6.54
1,Produto 001,2024-03,21.31,24.76,-3.45,-13.93
2,Produto 001,2024-04,29.05,21.31,7.74,36.33
3,Produto 001,2024-05,22.86,29.05,-6.20,-21.32
4,Produto 001,2024-06,21.40,22.86,-1.46,-6.39
...,...,...,...,...,...,...
5080,Produto 300,2025-08,45.48,43.51,1.97,4.53
5081,Produto 300,2025-09,42.19,45.48,-3.29,-7.23
5082,Produto 300,2025-10,44.78,42.19,2.59,6.13
5083,Produto 300,2025-11,37.98,44.78,-6.79,-15.16


### 2.6 — Pontualidade e completude por pedido

#### 2.6.1 — Indicadores de pontualidade e completude por pedido


In [ ]:
query = """
WITH pedidos_itens AS (

    SELECT
        p.pedido_id,
        p.fornecedor_id,
        p.data_pedido,
        p.data_prevista_entrega,

        SUM(i.quantidade) AS quantidade_pedida

    FROM pedidos_compra p

    JOIN itens_pedido i
        ON p.pedido_id = i.pedido_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        p.pedido_id,
        p.fornecedor_id,
        p.data_pedido,
        p.data_prevista_entrega
),

pedidos_entregas AS (

    SELECT
        pedido_id,

        SUM(quantidade_entregue) AS quantidade_entregue,

        MAX(data_entrega) AS data_ultima_entrega

    FROM entregas

    GROUP BY pedido_id
),

analise_otif AS (

    SELECT
        pi.pedido_id,
        pi.fornecedor_id,

        pi.quantidade_pedida,

        COALESCE(
            pe.quantidade_entregue,
            0
        ) AS quantidade_entregue,

        pi.data_prevista_entrega,

        pe.data_ultima_entrega,

        CASE
            WHEN
                COALESCE(pe.quantidade_entregue, 0)
                >= pi.quantidade_pedida
            THEN 1
            ELSE 0
        END AS in_full,

        CASE
            WHEN
                pe.data_ultima_entrega <= pi.data_prevista_entrega
            THEN 1
            ELSE 0
        END AS on_time

    FROM pedidos_itens pi

    LEFT JOIN pedidos_entregas pe
        ON pi.pedido_id = pe.pedido_id
)

SELECT
    f.nome_fornecedor AS fornecedor,

    COUNT(ao.pedido_id) AS total_pedidos,

    SUM(ao.in_full) AS pedidos_in_full,

    SUM(ao.on_time) AS pedidos_on_time,

    SUM(
        CASE
            WHEN ao.in_full = 1
             AND ao.on_time = 1
            THEN 1
            ELSE 0
        END
    ) AS pedidos_otif,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN ao.in_full = 1
                 AND ao.on_time = 1
                THEN 1
                ELSE 0
            END
        )
        / COUNT(ao.pedido_id),
        2
    ) AS percentual_otif

FROM analise_otif ao

JOIN fornecedores f
    ON ao.fornecedor_id = f.fornecedor_id

GROUP BY
    f.fornecedor_id,
    f.nome_fornecedor

ORDER BY
    percentual_otif DESC;
"""

pd.read_sql_query(query, conn)

,fornecedor,total_pedidos,pedidos_in_full,pedidos_on_time,pedidos_otif,percentual_otif
0,Fornecedor 12,178,170,154,149,83.71
1,Fornecedor 49,94,90,80,78,82.98
2,Fornecedor 16,148,142,124,122,82.43
3,Fornecedor 28,187,175,156,154,82.35
4,Fornecedor 40,129,125,108,106,82.17
5,Fornecedor 41,114,107,97,93,81.58
6,Fornecedor 17,223,212,182,180,80.72
7,Fornecedor 22,165,162,134,133,80.61
8,Fornecedor 34,93,86,78,74,79.57
9,Fornecedor 23,141,130,116,112,79.43


#### 2.6.2 — Pedidos entregues completos e pontualmente (OTIF)


In [ ]:
query = """
WITH pedidos_itens AS (

    SELECT
        p.pedido_id,
        p.fornecedor_id,

        SUM(i.quantidade) AS quantidade_pedida,

        p.data_prevista_entrega

    FROM pedidos_compra p

    JOIN itens_pedido i
        ON p.pedido_id = i.pedido_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        p.pedido_id,
        p.fornecedor_id,
        p.data_prevista_entrega
),

pedidos_entregas AS (

    SELECT
        pedido_id,

        SUM(quantidade_entregue) AS quantidade_entregue,

        MAX(data_entrega) AS data_ultima_entrega

    FROM entregas

    GROUP BY pedido_id
),

otif_pedidos AS (

    SELECT
        pi.pedido_id,
        pi.fornecedor_id,

        CASE
            WHEN
                COALESCE(pe.quantidade_entregue, 0)
                >= pi.quantidade_pedida
            AND
                pe.data_ultima_entrega <= pi.data_prevista_entrega
            THEN 1
            ELSE 0
        END AS otif

    FROM pedidos_itens pi

    LEFT JOIN pedidos_entregas pe
        ON pi.pedido_id = pe.pedido_id
),

desempenho_fornecedor AS (

    SELECT
        fornecedor_id,

        COUNT(pedido_id) AS quantidade_pedidos,

        SUM(otif) AS pedidos_otif,

        ROUND(
            100.0 * SUM(otif)
            / COUNT(pedido_id),
            2
        ) AS percentual_otif

    FROM otif_pedidos

    GROUP BY fornecedor_id
),

gasto_fornecedor AS (

    SELECT
        p.fornecedor_id,

        ROUND(
            SUM(i.quantidade * i.preco_unitario),
            2
        ) AS gasto_total

    FROM pedidos_compra p

    JOIN itens_pedido i
        ON p.pedido_id = i.pedido_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY p.fornecedor_id
)

SELECT
    f.nome_fornecedor AS fornecedor,

    df.quantidade_pedidos,

    df.pedidos_otif,

    df.percentual_otif,

    gf.gasto_total,

    RANK() OVER (
        ORDER BY df.percentual_otif DESC
    ) AS ranking_otif

FROM desempenho_fornecedor df

JOIN gasto_fornecedor gf
    ON df.fornecedor_id = gf.fornecedor_id

JOIN fornecedores f
    ON df.fornecedor_id = f.fornecedor_id

ORDER BY
    ranking_otif,
    gf.gasto_total DESC;
"""

pd.read_sql_query(query, conn)

,fornecedor,quantidade_pedidos,pedidos_otif,percentual_otif,gasto_total,ranking_otif
0,Fornecedor 12,178,149,83.71,3076558.69,1
1,Fornecedor 49,94,78,82.98,1115009.89,2
2,Fornecedor 16,148,122,82.43,1472881.10,3
3,Fornecedor 28,187,154,82.35,4255235.25,4
4,Fornecedor 40,129,106,82.17,825505.16,5
5,Fornecedor 41,114,93,81.58,886008.69,6
6,Fornecedor 17,223,180,80.72,5679982.84,7
7,Fornecedor 22,165,133,80.61,1492860.99,8
8,Fornecedor 34,93,74,79.57,1102544.52,9
9,Fornecedor 23,141,112,79.43,5953974.18,10


### 2.7 — Indicadores complementares de produtos e fornecedores

#### 2.7.1 — Oportunidades de economia por produto


In [ ]:
query = """
WITH resumo_produto AS (

    SELECT
        pr.produto_id,
        pr.nome_produto AS produto,

        SUM(i.quantidade) AS quantidade_total,

        SUM(
            i.quantidade * i.preco_unitario
        ) AS gasto_total,

        AVG(i.preco_unitario) AS preco_medio,

        MIN(i.preco_unitario) AS menor_preco

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        pr.produto_id,
        pr.nome_produto
),

oportunidades AS (

    SELECT
        produto_id,
        produto,
        quantidade_total,
        gasto_total,
        preco_medio,
        menor_preco,

        preco_medio - menor_preco
            AS diferenca_unitaria,

        quantidade_total
        * (preco_medio - menor_preco)
            AS economia_potencial

    FROM resumo_produto

    WHERE menor_preco < preco_medio
)

SELECT
    produto,

    quantidade_total,

    ROUND(
        gasto_total,
        2
    ) AS gasto_total,

    ROUND(
        preco_medio,
        2
    ) AS preco_medio,

    ROUND(
        menor_preco,
        2
    ) AS menor_preco,

    ROUND(
        diferenca_unitaria,
        2
    ) AS diferenca_unitaria,

    ROUND(
        economia_potencial,
        2
    ) AS economia_potencial

FROM oportunidades

ORDER BY
    economia_potencial DESC

LIMIT 20;
"""

pd.read_sql_query(query, conn)

,produto,quantidade_total,gasto_total,preco_medio,menor_preco,diferenca_unitaria,economia_potencial
0,Produto 178,13900,7476785.69,533.23,233.59,299.64,4165039.39
1,Produto 126,17603,6344395.72,360.11,190.09,170.02,2992870.06
2,Produto 086,16883,5487712.46,325.69,163.99,161.70,2730018.01
3,Produto 291,13677,4173884.83,306.94,137.18,169.76,2321812.38
4,Produto 242,10704,4368701.41,404.22,195.46,208.76,2234615.88
5,Produto 054,26734,4443137.52,165.79,84.63,81.16,2169618.33
6,Produto 045,23193,3660808.33,156.86,69.64,87.22,2022945.81
7,Produto 272,9043,3893212.62,426.66,219.42,207.24,1874063.73
8,Produto 252,5887,3310324.50,562.79,245.23,317.56,1869459.97
9,Produto 245,5322,3601450.30,654.83,341.23,313.60,1669002.25


#### 2.7.2 — Diversificação de fornecedores por produto


In [ ]:
query = """
WITH resumo_produto AS (

    SELECT
        pr.produto_id,
        pr.nome_produto AS produto,

        COUNT(DISTINCT p.fornecedor_id)
            AS quantidade_fornecedores,

        COUNT(DISTINCT p.pedido_id)
            AS quantidade_pedidos,

        SUM(i.quantidade)
            AS quantidade_total,

        SUM(
            i.quantidade * i.preco_unitario
        ) AS gasto_total

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        pr.produto_id,
        pr.nome_produto
),

total_gasto AS (

    SELECT
        SUM(gasto_total) AS gasto_empresa

    FROM resumo_produto
)

SELECT
    rp.produto,

    rp.quantidade_fornecedores,

    rp.quantidade_pedidos,

    rp.quantidade_total,

    ROUND(
        rp.gasto_total,
        2
    ) AS gasto_total,

    ROUND(
        100.0 * rp.gasto_total
        / tg.gasto_empresa,
        2
    ) AS percentual_gasto,

    CASE
        WHEN rp.quantidade_fornecedores = 1
             AND rp.gasto_total >= 500000
            THEN 'ALTO RISCO'

        WHEN rp.quantidade_fornecedores <= 2
             AND rp.gasto_total >= 500000
            THEN 'RISCO MODERADO'

        WHEN rp.quantidade_fornecedores = 1
            THEN 'DEPENDÊNCIA'

        ELSE 'BAIXO RISCO'
    END AS classificacao_risco

FROM resumo_produto rp

CROSS JOIN total_gasto tg

ORDER BY
    rp.gasto_total DESC;
"""

pd.read_sql_query(query, conn)

,produto,quantidade_fornecedores,quantidade_pedidos,quantidade_total,gasto_total,percentual_gasto,classificacao_risco
0,Produto 178,8,173,13900,7476785.69,5.23,BAIXO RISCO
1,Produto 126,7,264,17603,6344395.72,4.44,BAIXO RISCO
2,Produto 086,4,247,16883,5487712.46,3.84,BAIXO RISCO
3,Produto 054,8,390,26734,4443137.52,3.11,BAIXO RISCO
4,Produto 242,6,160,10704,4368701.41,3.06,BAIXO RISCO
...,...,...,...,...,...,...,...
294,Produto 279,1,1,19,992.94,0.00,DEPENDÊNCIA
295,Produto 255,7,17,266,915.37,0.00,BAIXO RISCO
296,Produto 241,3,3,46,829.31,0.00,BAIXO RISCO
297,Produto 065,3,7,88,732.60,0.00,BAIXO RISCO


#### 2.7.3 — Indicadores de desempenho dos fornecedores


In [ ]:
query = """
WITH gasto_fornecedor AS (

    SELECT
        p.fornecedor_id,

        SUM(
            i.quantidade * i.preco_unitario
        ) AS gasto_total,

        COUNT(DISTINCT p.pedido_id)
            AS quantidade_pedidos

    FROM pedidos_compra p

    JOIN itens_pedido i
        ON p.pedido_id = i.pedido_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY p.fornecedor_id
),

otif_pedidos AS (

    SELECT
        p.pedido_id,
        p.fornecedor_id,

        SUM(i.quantidade) AS quantidade_pedida,

        p.data_prevista_entrega,

        MAX(e.data_entrega) AS data_ultima_entrega,

        SUM(e.quantidade_entregue) AS quantidade_entregue

    FROM pedidos_compra p

    JOIN itens_pedido i
        ON p.pedido_id = i.pedido_id

    LEFT JOIN entregas e
        ON p.pedido_id = e.pedido_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        p.pedido_id,
        p.fornecedor_id,
        p.data_prevista_entrega
),

desempenho_fornecedor AS (

    SELECT
        fornecedor_id,

        COUNT(pedido_id) AS pedidos_analisados,

        SUM(
            CASE
                WHEN quantidade_entregue >= quantidade_pedida
                 AND data_ultima_entrega <= data_prevista_entrega
                THEN 1
                ELSE 0
            END
        ) AS pedidos_otif,

        ROUND(
            100.0 *
            SUM(
                CASE
                    WHEN quantidade_entregue >= quantidade_pedida
                     AND data_ultima_entrega <= data_prevista_entrega
                    THEN 1
                    ELSE 0
                END
            )
            / COUNT(pedido_id),
            2
        ) AS percentual_otif

    FROM otif_pedidos

    GROUP BY fornecedor_id
),

consolidado AS (

    SELECT
        f.fornecedor_id,
        f.nome_fornecedor AS fornecedor,

        gf.gasto_total,

        gf.quantidade_pedidos,

        ROUND(
            gf.gasto_total
            / gf.quantidade_pedidos,
            2
        ) AS valor_medio_pedido,

        df.percentual_otif

    FROM fornecedores f

    JOIN gasto_fornecedor gf
        ON f.fornecedor_id = gf.fornecedor_id

    JOIN desempenho_fornecedor df
        ON f.fornecedor_id = df.fornecedor_id
),

total_empresa AS (

    SELECT
        SUM(gasto_total) AS gasto_total_empresa

    FROM consolidado
)

SELECT
    c.fornecedor,

    c.quantidade_pedidos,

    ROUND(
        c.gasto_total,
        2
    ) AS gasto_total,

    ROUND(
        100.0 * c.gasto_total
        / te.gasto_total_empresa,
        2
    ) AS percentual_gasto,

    c.valor_medio_pedido,

    c.percentual_otif,

    RANK() OVER (
        ORDER BY c.gasto_total DESC
    ) AS ranking_gasto,

    RANK() OVER (
        ORDER BY c.percentual_otif DESC
    ) AS ranking_otif

FROM consolidado c

CROSS JOIN total_empresa te

ORDER BY
    c.gasto_total DESC;
"""

pd.read_sql_query(query, conn)

,fornecedor,quantidade_pedidos,gasto_total,percentual_gasto,valor_medio_pedido,percentual_otif,ranking_gasto,ranking_otif
0,Fornecedor 04,506,11088800.96,7.76,21914.63,68.97,1,21
1,Fornecedor 05,361,10521630.42,7.36,29145.79,54.57,2,47
2,Fornecedor 10,305,9562174.88,6.69,31351.39,65.57,3,32
3,Fornecedor 02,402,8134214.60,5.69,20234.36,52.99,4,50
4,Fornecedor 09,299,7161608.40,5.01,23951.87,53.85,5,49
5,Fornecedor 50,206,5996246.36,4.19,29107.99,67.96,6,24
6,Fornecedor 23,141,5953974.18,4.16,42226.77,74.47,7,8
7,Fornecedor 17,223,5679982.84,3.97,25470.78,74.89,8,7
8,Fornecedor 01,362,5515668.37,3.86,15236.65,55.52,9,45
9,Fornecedor 03,260,5450087.12,3.81,20961.87,54.23,10,48


## 3. Modelagem analítica com views SQL

### 3.1 — View de pedidos analíticos: `vw_pedidos_analiticos`


**Granularidade:** uma linha por pedido. Itens e eventos de entrega são agregados separadamente antes do relacionamento para evitar multiplicação de linhas.

#### Criação da view `vw_pedidos_analiticos`


In [ ]:
query = """
DROP VIEW IF EXISTS vw_pedidos_analiticos;

CREATE VIEW vw_pedidos_analiticos AS

WITH itens AS (
    SELECT
        pedido_id,
        SUM(quantidade) AS quantidade_pedida,
        SUM(quantidade * preco_unitario) AS valor_pedido
    FROM itens_pedido
    GROUP BY pedido_id
),

entregas_agrupadas AS (
    SELECT
        pedido_id,
        SUM(quantidade_entregue) AS quantidade_entregue,
        MAX(data_entrega) AS data_ultima_entrega
    FROM entregas
    GROUP BY pedido_id
)

SELECT
    p.pedido_id,
    p.fornecedor_id,
    f.nome_fornecedor,

    p.data_pedido,
    p.data_prevista_entrega,

    itens.quantidade_pedida,
    COALESCE(
        entregas_agrupadas.quantidade_entregue,
        0
    ) AS quantidade_entregue,

    entregas_agrupadas.data_ultima_entrega,

    ROUND(
        julianday(entregas_agrupadas.data_ultima_entrega)
        - julianday(p.data_pedido),
        2
    ) AS prazo_real_dias,

    ROUND(
        julianday(p.data_prevista_entrega)
        - julianday(p.data_pedido),
        2
    ) AS prazo_planejado_dias,

    CASE
        WHEN entregas_agrupadas.data_ultima_entrega IS NOT NULL
        THEN ROUND(
            julianday( entregas_agrupadas.data_ultima_entrega )
            - julianday( p.data_prevista_entrega ),
            2
        )
        ELSE NULL
    END AS dias_atraso,

    CASE
        WHEN COALESCE(
            entregas_agrupadas.quantidade_entregue,
            0
        ) >= itens.quantidade_pedida
        THEN 1
        ELSE 0
    END AS in_full,

    CASE
        WHEN entregas_agrupadas.data_ultima_entrega IS NOT NULL
             AND entregas_agrupadas.data_ultima_entrega
                 <= p.data_prevista_entrega
        THEN 1
        ELSE 0
    END AS on_time,

    CASE
        WHEN COALESCE(
            entregas_agrupadas.quantidade_entregue,
            0
        ) >= itens.quantidade_pedida
             AND entregas_agrupadas.data_ultima_entrega IS NOT NULL
             AND entregas_agrupadas.data_ultima_entrega
                 <= p.data_prevista_entrega
        THEN 1
        ELSE 0
    END AS otif,

    itens.valor_pedido,
    p.status_pedido

FROM pedidos_compra p

JOIN fornecedores f
    ON p.fornecedor_id = f.fornecedor_id

JOIN itens
    ON p.pedido_id = itens.pedido_id

LEFT JOIN entregas_agrupadas
    ON p.pedido_id = entregas_agrupadas.pedido_id;
"""

conn.executescript(query)

print("View criada com sucesso!")

View criada com sucesso!


#### Validação 3.1.1 — amostra de registros


In [ ]:
pd.read_sql_query("""
SELECT *
FROM vw_pedidos_analiticos
LIMIT 10;
""", conn)

,pedido_id,fornecedor_id,nome_fornecedor,data_pedido,data_prevista_entrega,quantidade_pedida,quantidade_entregue,data_ultima_entrega,prazo_real_dias,prazo_planejado_dias,dias_atraso,in_full,on_time,otif,valor_pedido,status_pedido
0,1,34,Fornecedor 34,2025-04-27,2025-05-03,124,124,2025-05-03,6.0,6.0,0.0,1,1,1,15449.49,Concluído
1,10,4,Fornecedor 04,2025-10-20,2025-10-28,180,180,2025-10-28,8.0,8.0,0.0,1,1,1,31785.00,Em Aberto
2,100,11,Fornecedor 11,2025-02-05,2025-02-09,56,56,2025-02-09,4.0,4.0,0.0,1,1,1,4003.04,Concluído
3,1000,24,Fornecedor 24,2025-02-22,2025-02-26,152,152,2025-02-26,4.0,4.0,0.0,1,1,1,17993.28,Concluído
4,1001,24,Fornecedor 24,2025-08-28,2025-09-03,6,6,2025-09-03,6.0,6.0,0.0,1,1,1,104.10,Concluído
5,1002,18,Fornecedor 18,2025-02-07,2025-02-17,89,89,2025-02-17,10.0,10.0,0.0,1,1,1,3049.14,Concluído
6,1003,48,Fornecedor 48,2024-05-30,2024-06-03,84,84,2024-06-03,4.0,4.0,0.0,1,1,1,6784.20,Concluído
7,1004,49,Fornecedor 49,2025-02-25,2025-03-03,264,264,2025-03-03,6.0,6.0,0.0,1,1,1,14899.78,Concluído
8,1005,38,Fornecedor 38,2025-02-17,2025-02-25,109,109,2025-02-25,8.0,8.0,0.0,1,1,1,2809.36,Concluído
9,1006,8,Fornecedor 08,2025-06-27,2025-07-10,19,19,2025-07-10,13.0,13.0,0.0,1,1,1,424.08,Concluído


#### Validação 3.1.2 — consistência da granularidade


In [ ]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS quantidade_linhas,
    COUNT(DISTINCT pedido_id) AS pedidos_unicos
FROM vw_pedidos_analiticos;
""", conn)

,quantidade_linhas,pedidos_unicos
0,8000,8000


### 3.2 — View de desempenho de fornecedores: `vw_desempenho_fornecedores`


**Granularidade:** uma linha por fornecedor, com gasto e indicadores agregados de desempenho.

#### Criação da view `vw_desempenho_fornecedores`


In [ ]:
query = """
DROP VIEW IF EXISTS vw_desempenho_fornecedores;

CREATE VIEW vw_desempenho_fornecedores AS

WITH total_empresa AS (
    SELECT
        SUM(valor_pedido) AS gasto_total_empresa
    FROM vw_pedidos_analiticos
    WHERE status_pedido <> 'Cancelado'
),

desempenho AS (
    SELECT
        fornecedor_id,
        nome_fornecedor,

        COUNT(DISTINCT pedido_id) AS quantidade_pedidos,

        SUM(valor_pedido) AS gasto_total,

        AVG(valor_pedido) AS gasto_medio_pedido,

        AVG(otif) * 100 AS percentual_otif,

        AVG(on_time) * 100 AS percentual_on_time,

        AVG(in_full) * 100 AS percentual_in_full

    FROM vw_pedidos_analiticos

    WHERE status_pedido <> 'Cancelado'

    GROUP BY
        fornecedor_id,
        nome_fornecedor
)

SELECT
    d.fornecedor_id,
    d.nome_fornecedor,

    d.quantidade_pedidos,

    ROUND(d.gasto_total, 2) AS gasto_total,

    ROUND(d.gasto_medio_pedido, 2) AS gasto_medio_pedido,

    ROUND(
        d.gasto_total / t.gasto_total_empresa * 100,
        2
    ) AS percentual_gasto_empresa,

    ROUND(d.percentual_otif, 2) AS percentual_otif,

    ROUND(d.percentual_on_time, 2) AS percentual_on_time,

    ROUND(d.percentual_in_full, 2) AS percentual_in_full

FROM desempenho d

CROSS JOIN total_empresa t;
"""

conn.executescript(query)

print("View criada com sucesso!")

View criada com sucesso!


#### Validação 3.2.1 — amostra de registros


In [ ]:
pd.read_sql_query("""
SELECT *
FROM vw_desempenho_fornecedores
ORDER BY gasto_total DESC
LIMIT 10;
""", conn)

,fornecedor_id,nome_fornecedor,quantidade_pedidos,gasto_total,gasto_medio_pedido,percentual_gasto_empresa,percentual_otif,percentual_on_time,percentual_in_full
0,4,Fornecedor 04,506,11088800.96,21914.63,7.76,75.69,78.06,95.06
1,5,Fornecedor 05,361,10521630.42,29145.79,7.36,59.83,62.33,92.80
2,10,Fornecedor 10,305,9562174.88,31351.39,6.69,70.49,72.46,95.08
3,2,Fornecedor 02,402,8134214.60,20234.36,5.69,59.45,61.44,94.03
4,9,Fornecedor 09,299,7161608.40,23951.87,5.01,58.86,60.87,93.65
5,50,Fornecedor 50,206,5996246.36,29107.99,4.19,76.21,76.70,95.63
6,23,Fornecedor 23,141,5953974.18,42226.77,4.16,79.43,82.27,92.20
7,17,Fornecedor 17,223,5679982.84,25470.78,3.97,80.72,81.61,95.07
8,1,Fornecedor 01,362,5515668.37,15236.65,3.86,61.88,64.36,92.82
9,3,Fornecedor 03,260,5450087.12,20961.87,3.81,61.15,62.69,93.85


#### Validação 3.2.2 — consistência da granularidade


In [ ]:
resultado = pd.read_sql_query("""
SELECT
    COUNT(*) AS quantidade_fornecedores,
    SUM(gasto_total) AS gasto_total
FROM vw_desempenho_fornecedores;
""", conn)

resultado["gasto_total"] = resultado["gasto_total"].map(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

resultado

,quantidade_fornecedores,gasto_total
0,50,"R$ 142.979.742,46"


### 3.3 — View de oportunidades por produto: `vw_oportunidades_produtos`


**Granularidade:** uma linha por produto com compras válidas. A economia potencial compara o preço médio observado com o menor preço observado.

#### Criação da view `vw_oportunidades_produtos`


In [ ]:
query = """
DROP VIEW IF EXISTS vw_oportunidades_produtos;

CREATE VIEW vw_oportunidades_produtos AS

WITH analise_produtos AS (

    SELECT
        i.produto_id,
        pr.nome_produto,
        pr.categoria_id,

        SUM(i.quantidade) AS quantidade_total,

        SUM(i.quantidade * i.preco_unitario) AS gasto_total,

        AVG(i.preco_unitario) AS preco_medio,

        MIN(i.preco_unitario) AS menor_preco

    FROM itens_pedido i

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        i.produto_id,
        pr.nome_produto,
        pr.categoria_id
)

SELECT
    produto_id,
    nome_produto,
    categoria_id,

    quantidade_total,

    ROUND(gasto_total, 2) AS gasto_total,

    ROUND(preco_medio, 2) AS preco_medio,

    ROUND(menor_preco, 2) AS menor_preco,

    ROUND(
        preco_medio - menor_preco,
        2
    ) AS diferenca_unitaria,

    ROUND(
        quantidade_total *
        (preco_medio - menor_preco),
        2
    ) AS economia_potencial

FROM analise_produtos;
"""

conn.executescript(query)

print("View criada com sucesso!")

View criada com sucesso!


#### Validação 3.3.1 — amostra de registros


In [ ]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS quantidade_linhas,
    COUNT(DISTINCT produto_id) AS produtos_unicos
FROM vw_oportunidades_produtos;
""", conn)

,quantidade_linhas,produtos_unicos
0,299,299


#### Validação complementar — produto sem compras válidas


In [ ]:
pd.read_sql_query("""
SELECT
    pr.produto_id,
    pr.nome_produto,
    pr.categoria_id
FROM produtos pr
LEFT JOIN vw_oportunidades_produtos op
    ON pr.produto_id = op.produto_id
WHERE op.produto_id IS NULL;
""", conn)

,produto_id,nome_produto,categoria_id
0,38,Produto 038,CAT002


#### Validação 3.3.2 — consistência da granularidade


In [ ]:
pd.read_sql_query("""
SELECT
    produto_id,
    nome_produto,
    quantidade_total,
    gasto_total,
    preco_medio,
    menor_preco,
    diferenca_unitaria,
    economia_potencial
FROM vw_oportunidades_produtos
ORDER BY economia_potencial DESC
LIMIT 10;
""", conn)

,produto_id,nome_produto,quantidade_total,gasto_total,preco_medio,menor_preco,diferenca_unitaria,economia_potencial
0,178,Produto 178,13900,7476785.69,533.23,233.59,299.64,4165039.39
1,126,Produto 126,17603,6344395.72,360.11,190.09,170.02,2992870.06
2,86,Produto 086,16883,5487712.46,325.69,163.99,161.70,2730018.01
3,291,Produto 291,13677,4173884.83,306.94,137.18,169.76,2321812.38
4,242,Produto 242,10704,4368701.41,404.22,195.46,208.76,2234615.88
5,54,Produto 054,26734,4443137.52,165.79,84.63,81.16,2169618.33
6,45,Produto 045,23193,3660808.33,156.86,69.64,87.22,2022945.81
7,272,Produto 272,9043,3893212.62,426.66,219.42,207.24,1874063.73
8,252,Produto 252,5887,3310324.50,562.79,245.23,317.56,1869459.97
9,245,Produto 245,5322,3601450.30,654.83,341.23,313.60,1669002.25


### 3.4 — View de gastos mensais: `vw_gastos_mensais`


**Granularidade:** uma linha por mês. O ticket médio é calculado a partir do valor agregado de cada pedido.

#### Criação da view `vw_gastos_mensais`


In [ ]:
query = """
DROP VIEW IF EXISTS vw_gastos_mensais;

CREATE VIEW vw_gastos_mensais AS

SELECT
    strftime('%Y-%m', p.data_pedido) AS mes,

    COUNT(DISTINCT p.pedido_id) AS quantidade_pedidos,

    SUM(i.quantidade) AS quantidade_total,

    SUM(i.quantidade * i.preco_unitario) AS gasto_total,

    AVG(i.preco_unitario) AS preco_medio_item,

    AVG(
        pedido_valor.valor_pedido
    ) AS valor_medio_pedido

FROM pedidos_compra p

JOIN itens_pedido i
    ON p.pedido_id = i.pedido_id

JOIN (
    SELECT
        pedido_id,
        SUM(quantidade * preco_unitario) AS valor_pedido
    FROM itens_pedido
    GROUP BY pedido_id
) AS pedido_valor
    ON p.pedido_id = pedido_valor.pedido_id

WHERE p.status_pedido <> 'Cancelado'

GROUP BY
    strftime('%Y-%m', p.data_pedido);
"""

conn.executescript(query)

print("View criada com sucesso!")

View criada com sucesso!


#### Validação 3.4.1 — amostra de registros


In [ ]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS quantidade_linhas,
    COUNT(DISTINCT mes) AS meses_unicos
FROM vw_gastos_mensais;
""", conn)

,quantidade_linhas,meses_unicos
0,24,24


#### Validação 3.4.2 — consistência da granularidade


In [ ]:
pd.read_sql_query("""
SELECT *
FROM vw_gastos_mensais
ORDER BY mes;
""", conn)

,mes,quantidade_pedidos,quantidade_total,gasto_total,preco_medio_item,valor_medio_pedido
0,2024-01,343,40081,5950307.33,151.347168,22052.376189
1,2024-02,313,38191,5581586.95,149.862370,22881.531369
2,2024-03,289,35595,4978936.12,150.419946,21170.620309
3,2024-04,309,35233,5007850.81,149.983560,20071.060288
4,2024-05,276,31380,4845458.37,161.695575,22175.655052
5,2024-06,314,39824,5618426.32,149.939723,22755.081083
6,2024-07,328,39073,6176902.78,169.472415,24117.742731
7,2024-08,335,43085,6286781.21,157.571012,22833.140267
8,2024-09,324,40376,6204650.18,156.673888,24249.216333
9,2024-10,340,41619,6381121.28,159.361632,23338.638505


### 3.5 — View de concentração por categoria e fornecedor: `vw_concentracao_categoria_fornecedor`


**Granularidade:** uma linha por combinação de categoria e fornecedor, para avaliar a concentração do gasto dentro de cada categoria.

#### Criação da view `vw_concentracao_categoria_fornecedor`


In [ ]:
query = """
DROP VIEW IF EXISTS vw_concentracao_categoria_fornecedor;

CREATE VIEW vw_concentracao_categoria_fornecedor AS

WITH gasto_categoria_fornecedor AS (

    SELECT
        c.categoria_id,
        c.nome_categoria,

        f.fornecedor_id,
        f.nome_fornecedor,

        SUM(i.quantidade * i.preco_unitario) AS gasto_fornecedor_categoria

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    JOIN categorias c
        ON pr.categoria_id = c.categoria_id

    JOIN fornecedores f
        ON p.fornecedor_id = f.fornecedor_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        c.categoria_id,
        c.nome_categoria,
        f.fornecedor_id,
        f.nome_fornecedor
),

total_categoria AS (

    SELECT
        categoria_id,
        SUM(gasto_fornecedor_categoria) AS gasto_total_categoria

    FROM gasto_categoria_fornecedor

    GROUP BY categoria_id
)

SELECT
    g.categoria_id,
    g.nome_categoria,

    g.fornecedor_id,
    g.nome_fornecedor,

    ROUND(
        g.gasto_fornecedor_categoria,
        2
    ) AS gasto_fornecedor_categoria,

    ROUND(
        t.gasto_total_categoria,
        2
    ) AS gasto_total_categoria,

    ROUND(
        g.gasto_fornecedor_categoria
        / t.gasto_total_categoria * 100,
        2
    ) AS percentual_categoria

FROM gasto_categoria_fornecedor g

JOIN total_categoria t
    ON g.categoria_id = t.categoria_id;
"""

conn.executescript(query)

print("View criada com sucesso!")

View criada com sucesso!


#### Validação 3.5.1 — amostra de registros


In [ ]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS quantidade_linhas,
    COUNT(DISTINCT categoria_id || '-' || fornecedor_id)
        AS combinacoes_unicas
FROM vw_concentracao_categoria_fornecedor;
""", conn)

,quantidade_linhas,combinacoes_unicas
0,431,431


#### Validação 3.5.2 — consistência da granularidade


In [ ]:
pd.read_sql_query("""
SELECT
    categoria_id,
    nome_categoria,
    ROUND(SUM(percentual_categoria), 2) AS percentual_total
FROM vw_concentracao_categoria_fornecedor
GROUP BY
    categoria_id,
    nome_categoria
ORDER BY categoria_id;
""", conn)

,categoria_id,nome_categoria,percentual_total
0,CAT001,Matéria-Prima,100.00
1,CAT002,Embalagens,100.01
2,CAT003,Componentes Industriais,100.01
3,CAT004,Produtos Químicos,100.02
4,CAT005,Material de Manutenção,100.00
5,CAT006,Equipamentos e Ferramentas,99.97
6,CAT007,Material de Escritório,100.01
7,CAT008,Higiene e Limpeza,100.02
8,CAT009,EPIs,100.00
9,CAT010,Outros Suprimentos,100.00


### 3.6 — View de risco de abastecimento: `vw_risco_abastecimento`


**Granularidade:** uma linha por produto com compras válidas. A classificação usa quantidade de fornecedores e um limite de gasto definido para a simulação.

#### Criação da view `vw_risco_abastecimento`


In [ ]:
query = """
DROP VIEW IF EXISTS vw_risco_abastecimento;

CREATE VIEW vw_risco_abastecimento AS

WITH analise_produtos AS (

    SELECT
        pr.produto_id,
        pr.nome_produto,
        pr.categoria_id,
        c.nome_categoria,

        COUNT(DISTINCT p.fornecedor_id)
            AS quantidade_fornecedores,

        COUNT(DISTINCT p.pedido_id)
            AS quantidade_pedidos,

        SUM(i.quantidade)
            AS quantidade_comprada,

        SUM(i.quantidade * i.preco_unitario)
            AS gasto_total

    FROM itens_pedido i

    JOIN pedidos_compra p
        ON i.pedido_id = p.pedido_id

    JOIN produtos pr
        ON i.produto_id = pr.produto_id

    JOIN categorias c
        ON pr.categoria_id = c.categoria_id

    WHERE p.status_pedido <> 'Cancelado'

    GROUP BY
        pr.produto_id,
        pr.nome_produto,
        pr.categoria_id,
        c.nome_categoria
),

total_empresa AS (

    SELECT
        SUM(gasto_total) AS gasto_total_empresa
    FROM analise_produtos
)

SELECT
    a.produto_id,
    a.nome_produto,
    a.categoria_id,
    a.nome_categoria,

    a.quantidade_fornecedores,
    a.quantidade_pedidos,
    a.quantidade_comprada,

    ROUND(a.gasto_total, 2) AS gasto_total,

    ROUND(
        a.gasto_total / t.gasto_total_empresa * 100,
        2
    ) AS percentual_gasto_empresa,

    CASE
        WHEN a.quantidade_fornecedores = 1
             AND a.gasto_total >= 500000
            THEN 'ALTO RISCO'

        WHEN a.quantidade_fornecedores <= 2
             AND a.gasto_total >= 500000
            THEN 'RISCO MODERADO'

        WHEN a.quantidade_fornecedores = 1
            THEN 'DEPENDÊNCIA DE FORNECEDOR'

        ELSE 'BAIXO RISCO'
    END AS classificacao_risco

FROM analise_produtos a

CROSS JOIN total_empresa t;
"""

conn.executescript(query)

print("View criada com sucesso!")

View criada com sucesso!


#### Validação 3.6.1 — amostra de registros


In [ ]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS quantidade_linhas,
    COUNT(DISTINCT produto_id) AS produtos_unicos
FROM vw_risco_abastecimento;
""", conn)

,quantidade_linhas,produtos_unicos
0,299,299


#### Validação 3.6.2 — consistência da granularidade


In [ ]:
pd.read_sql_query("""
SELECT
    produto_id,
    nome_produto,
    nome_categoria,
    quantidade_fornecedores,
    quantidade_pedidos,
    gasto_total,
    percentual_gasto_empresa,
    classificacao_risco
FROM vw_risco_abastecimento
ORDER BY
    CASE classificacao_risco
        WHEN 'ALTO RISCO' THEN 1
        WHEN 'RISCO MODERADO' THEN 2
        WHEN 'DEPENDÊNCIA DE FORNECEDOR' THEN 3
        ELSE 4
    END,
    gasto_total DESC
LIMIT 20;
""", conn)

,produto_id,nome_produto,nome_categoria,quantidade_fornecedores,quantidade_pedidos,gasto_total,percentual_gasto_empresa,classificacao_risco
0,208,Produto 208,Material de Manutenção,1,29,605458.24,0.42,ALTO RISCO
1,102,Produto 102,Componentes Industriais,2,86,1172026.97,0.82,RISCO MODERADO
2,207,Produto 207,Material de Escritório,2,58,998559.16,0.70,RISCO MODERADO
3,140,Produto 140,Componentes Industriais,2,30,861032.64,0.60,RISCO MODERADO
4,271,Produto 271,Matéria-Prima,2,70,841550.97,0.59,RISCO MODERADO
5,237,Produto 237,Material de Escritório,2,41,719897.17,0.50,RISCO MODERADO
6,278,Produto 278,Componentes Industriais,2,26,603322.22,0.42,RISCO MODERADO
7,49,Produto 049,Matéria-Prima,2,60,506634.49,0.35,RISCO MODERADO
8,100,Produto 100,Matéria-Prima,1,33,244695.00,0.17,DEPENDÊNCIA DE FORNECEDOR
9,125,Produto 125,Produtos Químicos,1,14,27671.57,0.02,DEPENDÊNCIA DE FORNECEDOR


### 3.7 — View de scorecard executivo: `vw_scorecard_executivo`


**Granularidade:** uma linha por fornecedor, combinando exposição financeira e indicadores de desempenho.

#### Criação da view `vw_scorecard_executivo`


In [ ]:
query = """
DROP VIEW IF EXISTS vw_scorecard_executivo;

CREATE VIEW vw_scorecard_executivo AS

WITH total_empresa AS (
    SELECT
        SUM(valor_pedido) AS gasto_total_empresa
    FROM vw_pedidos_analiticos
    WHERE status_pedido <> 'Cancelado'
),

scorecard AS (
    SELECT
        fornecedor_id,
        nome_fornecedor,

        COUNT(*) AS quantidade_pedidos,

        SUM(valor_pedido) AS gasto_total,

        AVG(valor_pedido) AS valor_medio_pedido,

        AVG(otif) * 100 AS percentual_otif,

        AVG(on_time) * 100 AS percentual_on_time,

        AVG(in_full) * 100 AS percentual_in_full

    FROM vw_pedidos_analiticos

    WHERE status_pedido <> 'Cancelado'

    GROUP BY
        fornecedor_id,
        nome_fornecedor
),

indicadores AS (
    SELECT
        s.*,

        s.gasto_total
            / t.gasto_total_empresa * 100
            AS percentual_gasto_empresa,

        RANK() OVER (
            ORDER BY s.gasto_total DESC
        ) AS ranking_gasto,

        RANK() OVER (
            ORDER BY s.percentual_otif DESC
        ) AS ranking_otif

    FROM scorecard s

    CROSS JOIN total_empresa t
)

SELECT
    fornecedor_id,
    nome_fornecedor,

    quantidade_pedidos,

    ROUND(gasto_total, 2) AS gasto_total,

    ROUND(percentual_gasto_empresa, 2)
        AS percentual_gasto_empresa,

    ROUND(valor_medio_pedido, 2)
        AS valor_medio_pedido,

    ROUND(percentual_otif, 2)
        AS percentual_otif,

    ROUND(percentual_on_time, 2)
        AS percentual_on_time,

    ROUND(percentual_in_full, 2)
        AS percentual_in_full,

    ranking_gasto,
    ranking_otif

FROM indicadores;
"""

conn.executescript(query)

print("View criada com sucesso!")

View criada com sucesso!


#### Validação 3.7.1 — amostra de registros


In [ ]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS quantidade_linhas,
    COUNT(DISTINCT fornecedor_id) AS fornecedores_unicos
FROM vw_scorecard_executivo;
""", conn)

,quantidade_linhas,fornecedores_unicos
0,50,50


#### Validação 3.7.2 — consistência da granularidade


In [ ]:
pd.read_sql_query("""
SELECT *
FROM vw_scorecard_executivo
ORDER BY ranking_gasto
LIMIT 10;
""", conn)

,fornecedor_id,nome_fornecedor,quantidade_pedidos,gasto_total,percentual_gasto_empresa,valor_medio_pedido,percentual_otif,percentual_on_time,percentual_in_full,ranking_gasto,ranking_otif
0,4,Fornecedor 04,506,11088800.96,7.76,21914.63,75.69,78.06,95.06,1,22
1,5,Fornecedor 05,361,10521630.42,7.36,29145.79,59.83,62.33,92.80,2,48
2,10,Fornecedor 10,305,9562174.88,6.69,31351.39,70.49,72.46,95.08,3,34
3,2,Fornecedor 02,402,8134214.60,5.69,20234.36,59.45,61.44,94.03,4,49
4,9,Fornecedor 09,299,7161608.40,5.01,23951.87,58.86,60.87,93.65,5,50
5,50,Fornecedor 50,206,5996246.36,4.19,29107.99,76.21,76.70,95.63,6,19
6,23,Fornecedor 23,141,5953974.18,4.16,42226.77,79.43,82.27,92.20,7,10
7,17,Fornecedor 17,223,5679982.84,3.97,25470.78,80.72,81.61,95.07,8,7
8,1,Fornecedor 01,362,5515668.37,3.86,15236.65,61.88,64.36,92.82,9,45
9,3,Fornecedor 03,260,5450087.12,3.81,20961.87,61.15,62.69,93.85,10,46


#### Validação 3.7.3 — consistência dos totais


In [ ]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS quantidade_fornecedores,
    ROUND(SUM(gasto_total), 2) AS gasto_total,
    MIN(ranking_gasto) AS menor_ranking_gasto,
    MIN(ranking_otif) AS menor_ranking_otif
FROM vw_scorecard_executivo;
""", conn)

,quantidade_fornecedores,gasto_total,menor_ranking_gasto,menor_ranking_otif
0,50,1.429797e+08,1,1


## 4. Dashboard executivo com Python e Plotly

O dashboard apresenta indicadores gerais e visualizações exploratórias. As análises detalhadas de desempenho, economia e risco aparecem nas seções seguintes.

#### Preparação das bibliotecas e dos dados do dashboard


In [ ]:
import pandas as pd

# Indicadores gerais dos pedidos
df_pedidos = pd.read_sql_query("""
    SELECT *
    FROM vw_pedidos_analiticos
""", conn)

# Gastos mensais
df_mensal = pd.read_sql_query("""
    SELECT *
    FROM vw_gastos_mensais
    ORDER BY mes
""", conn)

# Desempenho dos fornecedores
df_fornecedores = pd.read_sql_query("""
    SELECT *
    FROM vw_scorecard_executivo
    ORDER BY ranking_gasto
""", conn)

# Oportunidades de produtos
df_oportunidades = pd.read_sql_query("""
    SELECT *
    FROM vw_oportunidades_produtos
""", conn)

print("Dados carregados com sucesso!\n")

print(f"Pedidos: {len(df_pedidos):,}")
print(f"Meses: {len(df_mensal):,}")
print(f"Fornecedores: {len(df_fornecedores):,}")
print(f"Produtos com compras válidas: {len(df_oportunidades):,}")

display(df_pedidos.head())
display(df_mensal.head())
display(df_fornecedores.head())

Dados carregados com sucesso!

Pedidos: 8,000
Meses: 24
Fornecedores: 50
Produtos com compras válidas: 299


,pedido_id,fornecedor_id,nome_fornecedor,data_pedido,data_prevista_entrega,quantidade_pedida,quantidade_entregue,data_ultima_entrega,prazo_real_dias,prazo_planejado_dias,dias_atraso,in_full,on_time,otif,valor_pedido,status_pedido
0,1,34,Fornecedor 34,2025-04-27,2025-05-03,124,124,2025-05-03,6.0,6.0,0.0,1,1,1,15449.49,Concluído
1,10,4,Fornecedor 04,2025-10-20,2025-10-28,180,180,2025-10-28,8.0,8.0,0.0,1,1,1,31785.00,Em Aberto
2,100,11,Fornecedor 11,2025-02-05,2025-02-09,56,56,2025-02-09,4.0,4.0,0.0,1,1,1,4003.04,Concluído
3,1000,24,Fornecedor 24,2025-02-22,2025-02-26,152,152,2025-02-26,4.0,4.0,0.0,1,1,1,17993.28,Concluído
4,1001,24,Fornecedor 24,2025-08-28,2025-09-03,6,6,2025-09-03,6.0,6.0,0.0,1,1,1,104.10,Concluído


,mes,quantidade_pedidos,quantidade_total,gasto_total,preco_medio_item,valor_medio_pedido
0,2024-01,343,40081,5950307.33,151.347168,22052.376189
1,2024-02,313,38191,5581586.95,149.862370,22881.531369
2,2024-03,289,35595,4978936.12,150.419946,21170.620309
3,2024-04,309,35233,5007850.81,149.983560,20071.060288
4,2024-05,276,31380,4845458.37,161.695575,22175.655052


,fornecedor_id,nome_fornecedor,quantidade_pedidos,gasto_total,percentual_gasto_empresa,valor_medio_pedido,percentual_otif,percentual_on_time,percentual_in_full,ranking_gasto,ranking_otif
0,4,Fornecedor 04,506,11088800.96,7.76,21914.63,75.69,78.06,95.06,1,22
1,5,Fornecedor 05,361,10521630.42,7.36,29145.79,59.83,62.33,92.80,2,48
2,10,Fornecedor 10,305,9562174.88,6.69,31351.39,70.49,72.46,95.08,3,34
3,2,Fornecedor 02,402,8134214.60,5.69,20234.36,59.45,61.44,94.03,4,49
4,9,Fornecedor 09,299,7161608.40,5.01,23951.87,58.86,60.87,93.65,5,50


### 4.1 — Indicadores-chave de desempenho (KPIs)

In [ ]:
# Excluir pedidos cancelados da análise
df_validos = df_pedidos[
    df_pedidos["status_pedido"] != "Cancelado"
].copy()

# Calcular os indicadores executivos
gasto_total = df_validos["valor_pedido"].sum()

quantidade_pedidos = df_validos["pedido_id"].nunique()

percentual_otif = df_validos["otif"].mean() * 100

percentual_atrasados = (
    (df_validos["dias_atraso"] > 0).mean() * 100
)

valor_medio_pedido = (
    gasto_total / quantidade_pedidos
    if quantidade_pedidos > 0 else 0
)

# Exibir os resultados formatados
print("DASHBOARD EXECUTIVO — PROCUREMENT ANALYTICS")
print("-" * 48)
print(f"Gasto total: R$ {gasto_total:,.2f}")
print(f"Quantidade de pedidos: {quantidade_pedidos:,}")
print(f"OTIF: {percentual_otif:.2f}%")
print(f"Pedidos atrasados: {percentual_atrasados:.2f}%")
print(f"Valor médio por pedido: R$ {valor_medio_pedido:,.2f}")


DASHBOARD EXECUTIVO — PROCUREMENT ANALYTICS
------------------------------------------------
Gasto total: R$ 142,979,742.46
Quantidade de pedidos: 7,772
OTIF: 71.69%
Pedidos atrasados: 23.01%
Valor médio por pedido: R$ 18,396.78


### 4.2 — Visualizações do dashboard

#### 4.2.1 — Evolução mensal dos gastos com compras

In [ ]:
import plotly.express as px

print("Plotly importado com sucesso!")

Plotly importado com sucesso!


#### Preparação da série temporal mensal


In [ ]:
# Garantir que os meses estejam em ordem cronológica
df_mensal["mes"] = pd.to_datetime(
    df_mensal["mes"],
    format="%Y-%m"
)

# Criar o gráfico de evolução mensal
fig_gastos = px.line(
    df_mensal,
    x="mes",
    y="gasto_total",
    title="Evolução Mensal dos Gastos com Compras",
    markers=True,
    labels={
        "mes": "Mês",
        "gasto_total": "Gasto total (R$)"
    },
    hover_data={
        "quantidade_pedidos": True,
        "gasto_total": ":,.2f"
    }
)

fig_gastos.update_layout(
    xaxis_title="Mês",
    yaxis_title="Gasto total (R$)",
    template="plotly_white",
    hovermode="x unified"
)

fig_gastos.update_yaxes(
    tickprefix="R$ ",
    tickformat=",.0f"
)

fig_gastos.show()


#### 4.2.2 — Top 10 fornecedores por gasto

In [ ]:
df_top_fornecedores = (
    df_fornecedores
    .nlargest(10, "gasto_total")
    .sort_values("gasto_total", ascending=True)
)

fig_fornecedores = px.bar(
    df_top_fornecedores,
    x="gasto_total",
    y="nome_fornecedor",
    orientation="h",
    title="Top 10 Fornecedores por Gasto",
    labels={
        "gasto_total": "Gasto total (R$)",
        "nome_fornecedor": "Fornecedor"
    },
    hover_data={
        "quantidade_pedidos": True,
        "percentual_gasto_empresa": ":.2f",
        "percentual_otif": ":.2f"
    }
)

fig_fornecedores.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Gasto total (R$)",
    yaxis_title="Fornecedor",
    separators=",.",
    height=500
)

fig_fornecedores.update_xaxes(
    tickprefix="R$ ",
    tickformat=",.0f"
)

fig_fornecedores.update_traces(
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Gasto total: R$ %{x:,.2f}<br>"
        "<extra></extra>"
    )
)

fig_fornecedores.show()



#### 4.2.3 — Participação dos dez maiores fornecedores

In [ ]:
gasto_top10 = df_top_fornecedores["gasto_total"].sum()

gasto_empresa = df_fornecedores["gasto_total"].sum()

participacao_top10 = gasto_top10 / gasto_empresa * 100

print(f"Gasto dos 10 maiores fornecedores: R$ {gasto_top10:,.2f}")
print(f"Gasto total da empresa: R$ {gasto_empresa:,.2f}")
print(f"Participação dos 10 maiores: {participacao_top10:.2f}%")


Gasto dos 10 maiores fornecedores: R$ 75,064,388.13
Gasto total da empresa: R$ 142,979,742.46
Participação dos 10 maiores: 52.50%


#### 4.2.4 — Concentração dos gastos por fornecedor

In [ ]:
df_concentracao = pd.DataFrame({
    "Grupo": ["Top 10 fornecedores", "Demais fornecedores"],
    "Gasto": [
        gasto_top10,
        gasto_empresa - gasto_top10
    ]
})

fig_concentracao = px.pie(
    df_concentracao,
    names="Grupo",
    values="Gasto",
    hole=0.55,
    title="Concentração dos Gastos por Fornecedor",
    color="Grupo",
    color_discrete_map={
        "Top 10 fornecedores": "#636EFA",
        "Demais fornecedores": "#B8C2CC"
    }
)

fig_concentracao.update_traces(
    textposition="inside",
    texttemplate="%{percent:.1%}",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Gasto: R$ %{value:,.2f}<br>"
        "Participação: %{percent:.2%}"
        "<extra></extra>"
    )
)

fig_concentracao.update_layout(
    template="plotly_white",
    title_x=0.5,
    separators=",."
)

fig_concentracao.show()


### 4.3 — Página executiva provisória

In [ ]:
from IPython.display import display, HTML
import plotly.io as pio

kpis = [
    ("Gasto total", f"R$ {gasto_total:,.2f}"),
    ("Pedidos válidos", f"{quantidade_pedidos:,}"),
    ("OTIF", f"{percentual_otif:.2f}%"),
    ("Pedidos atrasados", f"{percentual_atrasados:.2f}%"),
    ("Ticket médio", f"R$ {valor_medio_pedido:,.2f}")
]

cards_html = ""

for titulo, valor in kpis:
    cards_html += f"""
    <div style="
        flex: 1;
        min-width: 150px;
        padding: 18px;
        border: 1px solid #D9DEE7;
        border-radius: 10px;
        background: #FFFFFF;
        box-sizing: border-box;
    ">
        <div style="font-size: 13px; color: #596579;">
            {titulo}
        </div>
        <div style="
            font-size: 23px;
            font-weight: 600;
            color: #243B64;
            margin-top: 10px;
        ">
            {valor}
        </div>
    </div>
    """

dashboard_html = f"""
<div style="font-family: Arial, sans-serif; color: #243B64;">
    <h1 style="font-size: 27px; margin-bottom: 4px;">
        Procurement Analytics
    </h1>
    <p style="color: #596579; margin-top: 0;">
        Nexa Distribuição Ltda. | Visão executiva | 2024–2025
    </p>
    <div style="
        display: flex;
        flex-wrap: wrap;
        gap: 12px;
        margin: 24px 0;
    ">
        {cards_html}
    </div>
</div>
"""

display(HTML(dashboard_html))

fig_gastos.show()
fig_fornecedores.show()
fig_concentracao.show()


#### Renderização do dashboard executivo


In [ ]:
from IPython.display import display, HTML

display(HTML("""
<h2 style="color:#243B64;">Procurement Analytics</h2>
<p style="color:#596579;">
Nexa Distribuição Ltda. | Visão executiva | 2024–2025
</p>
"""))

fig_gastos.show()
fig_fornecedores.show()
fig_concentracao.show()


## 5. Análise de desempenho dos fornecedores

### 5.1 — Relação entre gasto e desempenho dos fornecedores

In [ ]:
fig_desempenho = px.scatter(
    df_fornecedores,
    x="gasto_total",
    y="percentual_otif",
    size="quantidade_pedidos",
    hover_name="nome_fornecedor",
    hover_data={
        "gasto_total": ":,.2f",
        "percentual_otif": ":.2f",
        "quantidade_pedidos": True
    },
    title="Gasto x Desempenho dos Fornecedores",
    labels={
        "gasto_total": "Gasto total (R$)",
        "percentual_otif": "OTIF (%)",
        "quantidade_pedidos": "Quantidade de pedidos"
    }
)

fig_desempenho.update_layout(
    template="plotly_white",
    title_x=0.5,
    separators=",.",
    height=550
)

fig_desempenho.update_xaxes(
    tickprefix="R$ ",
    tickformat=",.0f"
)

fig_desempenho.show()


### 5.2 — Identificação dos fornecedores prioritários

In [ ]:
mediana_gasto = df_fornecedores["gasto_total"].median()
mediana_otif = df_fornecedores["percentual_otif"].median()

df_prioridades = df_fornecedores[
    (df_fornecedores["gasto_total"] > mediana_gasto) &
    (df_fornecedores["percentual_otif"] < mediana_otif)
].copy()

df_prioridades = df_prioridades.sort_values(
    "gasto_total",
    ascending=False
)

display(
    df_prioridades[
        [
            "nome_fornecedor",
            "quantidade_pedidos",
            "gasto_total",
            "percentual_gasto_empresa",
            "percentual_otif",
            "percentual_on_time",
            "percentual_in_full"
        ]
    ].round(2)
)

print(f"Mediana de gasto: R$ {mediana_gasto:,.2f}")
print(f"Mediana de OTIF: {mediana_otif:.2f}%")
print(f"Fornecedores prioritários identificados: {len(df_prioridades)}")


,nome_fornecedor,quantidade_pedidos,gasto_total,percentual_gasto_empresa,percentual_otif,percentual_on_time,percentual_in_full
1,Fornecedor 05,361,10521630.42,7.36,59.83,62.33,92.80
2,Fornecedor 10,305,9562174.88,6.69,70.49,72.46,95.08
3,Fornecedor 02,402,8134214.60,5.69,59.45,61.44,94.03
4,Fornecedor 09,299,7161608.40,5.01,58.86,60.87,93.65
8,Fornecedor 01,362,5515668.37,3.86,61.88,64.36,92.82
9,Fornecedor 03,260,5450087.12,3.81,61.15,62.69,93.85
10,Fornecedor 08,281,4504765.69,3.15,61.92,63.35,92.88
11,Fornecedor 32,100,4454116.23,3.12,67.00,69.00,93.00
14,Fornecedor 19,88,3237153.05,2.26,72.73,75.00,95.45
15,Fornecedor 38,126,3179382.79,2.22,73.81,74.60,93.65


Mediana de gasto: R$ 1,471,092.46
Mediana de OTIF: 74.47%
Fornecedores prioritários identificados: 12


### 5.3 — Impacto financeiro e operacional dos fornecedores prioritários


In [ ]:
gasto_prioritarios = df_prioridades["gasto_total"].sum()

participacao_prioritarios = (
    gasto_prioritarios / gasto_empresa * 100
)

pedidos_prioritarios = df_prioridades["quantidade_pedidos"].sum()

print(f"Fornecedores prioritários: {len(df_prioridades)}")
print(f"Gasto associado ao grupo: R$ {gasto_prioritarios:,.2f}")
print(f"Participação no gasto total: {participacao_prioritarios:.2f}%")
print(f"Pedidos associados ao grupo: {pedidos_prioritarios:,}")

print(
    f"OTIF médio do grupo: "
    f"{df_prioridades['percentual_otif'].mean():.2f}%"
)

print(
    f"OTIF médio dos 50 fornecedores: "
    f"{df_fornecedores['percentual_otif'].mean():.2f}%"
)

Fornecedores prioritários: 12
Gasto associado ao grupo: R$ 67,535,677.09
Participação no gasto total: 47.23%
Pedidos associados ao grupo: 2,832
OTIF médio do grupo: 65.91%
OTIF médio dos 50 fornecedores: 73.10%


### 5.4 — Comparação de OTIF e pontualidade por pedido


In [ ]:

ids_prioritarios = df_prioridades["fornecedor_id"].tolist()

df_pedidos_prioritarios = df_validos[
    df_validos["fornecedor_id"].isin(ids_prioritarios)
]

df_pedidos_demais = df_validos[
    ~df_validos["fornecedor_id"].isin(ids_prioritarios)
]

otif_prioritarios = df_pedidos_prioritarios["otif"].mean() * 100
otif_demais = df_pedidos_demais["otif"].mean() * 100

on_time_prioritarios = (
    df_pedidos_prioritarios["on_time"].mean() * 100
)

on_time_demais = df_pedidos_demais["on_time"].mean() * 100

print(f"OTIF ponderado por pedido — grupo prioritário: {otif_prioritarios:.2f}%")
print(f"OTIF ponderado por pedido — demais fornecedores: {otif_demais:.2f}%")
print(f"On Time — grupo prioritário: {on_time_prioritarios:.2f}%")
print(f"On Time — demais fornecedores: {on_time_demais:.2f}%")

OTIF ponderado por pedido — grupo prioritário: 63.81%
OTIF ponderado por pedido — demais fornecedores: 76.21%
On Time — grupo prioritário: 65.82%
On Time — demais fornecedores: 78.16%


### 5.5 — Comparação de OTIF entre fornecedores prioritários


In [ ]:

df_otif_prioritarios = df_prioridades.sort_values(
    "percentual_otif",
    ascending=True
).copy()

fig_otif_prioritarios = px.bar(
    df_otif_prioritarios,
    x="percentual_otif",
    y="nome_fornecedor",
    orientation="h",
    title="OTIF dos Fornecedores Prioritários",
    labels={
        "percentual_otif": "OTIF (%)",
        "nome_fornecedor": "Fornecedor"
    },
    hover_data={
        "gasto_total": ":,.2f",
        "quantidade_pedidos": True,
        "percentual_on_time": ":.2f",
        "percentual_in_full": ":.2f"
    }
)

fig_otif_prioritarios.update_layout(
    template="plotly_white",
    title_x=0.5,
    separators=",.",
    height=550,
    xaxis_title="OTIF (%)",
    yaxis_title="Fornecedor"
)

fig_otif_prioritarios.update_xaxes(
    ticksuffix="%",
    range=[0, 100]
)

fig_otif_prioritarios.update_traces(
    hovertemplate=(
        "<b>%{y}</b><br>"
        "OTIF: %{x:.2f}%<br>"
        "<extra></extra>"
    )
)

fig_otif_prioritarios.show()

### 5.6 — Componentes do OTIF: pontualidade e completude


In [ ]:
df_componentes_otif = df_prioridades[
    [
        "nome_fornecedor",
        "percentual_otif",
        "percentual_on_time",
        "percentual_in_full"
    ]
].sort_values("percentual_otif", ascending=True)

fig_componentes_otif = px.bar(
    df_componentes_otif,
    x="nome_fornecedor",
    y=[
        "percentual_otif",
        "percentual_on_time",
        "percentual_in_full"
    ],
    barmode="group",
    title="Componentes do OTIF por Fornecedor Prioritário",
    labels={
        "nome_fornecedor": "Fornecedor",
        "value": "Percentual (%)",
        "variable": "Indicador"
    },
    category_orders={
        "nome_fornecedor": df_componentes_otif[
            "nome_fornecedor"
        ].tolist()
    }
)

fig_componentes_otif.update_layout(
    template="plotly_white",
    title_x=0.5,
    height=600,
    xaxis_title="Fornecedor",
    yaxis_title="Percentual (%)",
    legend_title="Indicador"
)

fig_componentes_otif.update_yaxes(
    ticksuffix="%",
    range=[0, 100]
)

fig_componentes_otif.show()


### 5.7 — Diagnóstico dos três fornecedores mais críticos


In [ ]:
fornecedores_criticos = ["Fornecedor 05", "Fornecedor 02", "Fornecedor 09"]

df_diagnostico = (
    df_validos[
        df_validos["nome_fornecedor"].isin(fornecedores_criticos)
    ]
    .groupby("nome_fornecedor")
    .agg(
        quantidade_pedidos=("pedido_id", "nunique"),
        gasto_total=("valor_pedido", "sum"),
        percentual_otif=("otif", "mean"),
        percentual_on_time=("on_time", "mean"),
        percentual_in_full=("in_full", "mean"),
        percentual_atrasados=("dias_atraso", lambda x: (x > 0).mean())
    )
    .reset_index()
)

colunas_percentuais = [
    "percentual_otif",
    "percentual_on_time",
    "percentual_in_full",
    "percentual_atrasados"
]

df_diagnostico[colunas_percentuais] *= 100

display(
    df_diagnostico.sort_values("percentual_otif")
    .round(2)
)


,nome_fornecedor,quantidade_pedidos,gasto_total,percentual_otif,percentual_on_time,percentual_in_full,percentual_atrasados
2,Fornecedor 09,299,7161608.40,58.86,60.87,93.65,36.45
0,Fornecedor 02,402,8134214.60,59.45,61.44,94.03,35.32
1,Fornecedor 05,361,10521630.42,59.83,62.33,92.80,34.90


### 5.8 — Duração dos atrasos


In [ ]:

df_atrasos = df_validos[
    df_validos["nome_fornecedor"].isin(
        ["Fornecedor 05", "Fornecedor 02", "Fornecedor 09"]
    )
].copy()

df_atrasos = df_atrasos[df_atrasos["dias_atraso"] > 0]

resumo_atrasos = (
    df_atrasos
    .groupby("nome_fornecedor")
    .agg(
        pedidos_atrasados=("pedido_id", "nunique"),
        atraso_medio_dias=("dias_atraso", "mean"),
        atraso_maximo_dias=("dias_atraso", "max")
    )
    .reset_index()
    .sort_values("atraso_medio_dias", ascending=False)
)

display(resumo_atrasos.round(2))

,nome_fornecedor,pedidos_atrasados,atraso_medio_dias,atraso_maximo_dias
2,Fornecedor 09,109,5.76,33.0
1,Fornecedor 05,126,5.43,24.0
0,Fornecedor 02,142,5.20,35.0


### 5.9 — Evolução mensal dos atrasos


In [ ]:
df_atrasos_mensais = df_validos[
    df_validos["nome_fornecedor"].isin(
        ["Fornecedor 05", "Fornecedor 02", "Fornecedor 09"]
    )
].copy()

df_atrasos_mensais["mes"] = pd.to_datetime(
    df_atrasos_mensais["data_pedido"]
).dt.to_period("M").astype(str)

df_atrasos_mensais["pedido_atrasado"] = (
    df_atrasos_mensais["dias_atraso"] > 0
).astype(int)

resumo_mensal = (
    df_atrasos_mensais
    .groupby(["mes", "nome_fornecedor"])
    .agg(
        quantidade_pedidos=("pedido_id", "nunique"),
        pedidos_atrasados=("pedido_atrasado", "sum")
    )
    .reset_index()
)

resumo_mensal["percentual_atrasados"] = (
    resumo_mensal["pedidos_atrasados"]
    / resumo_mensal["quantidade_pedidos"] * 100
)

fig_atrasos_mensais = px.line(
    resumo_mensal,
    x="mes",
    y="percentual_atrasados",
    color="nome_fornecedor",
    markers=True,
    title="Evolução Mensal dos Atrasos por Fornecedor",
    labels={
        "mes": "Mês do pedido",
        "percentual_atrasados": "Pedidos atrasados (%)",
        "nome_fornecedor": "Fornecedor"
    },
    hover_data={
        "quantidade_pedidos": True,
        "pedidos_atrasados": True
    }
)

fig_atrasos_mensais.update_layout(
    template="plotly_white",
    title_x=0.5,
    height=500,
    yaxis_ticksuffix="%"
)

fig_atrasos_mensais.show()

### 5.10 — Meses com maiores taxas de atraso


In [ ]:
meses_criticos = (
    resumo_mensal
    .sort_values(
        ["percentual_atrasados", "quantidade_pedidos"],
        ascending=[False, False]
    )
    .head(10)
)

display(
    meses_criticos[
        [
            "mes",
            "nome_fornecedor",
            "quantidade_pedidos",
            "pedidos_atrasados",
            "percentual_atrasados"
        ]
    ].round(2)
)

,mes,nome_fornecedor,quantidade_pedidos,pedidos_atrasados,percentual_atrasados
41,2025-02,Fornecedor 09,15,9,60.00
29,2024-10,Fornecedor 09,12,7,58.33
27,2024-10,Fornecedor 02,16,9,56.25
48,2025-05,Fornecedor 02,16,9,56.25
16,2024-06,Fornecedor 05,18,10,55.56
33,2024-12,Fornecedor 02,9,5,55.56
7,2024-03,Fornecedor 05,11,6,54.55
15,2024-06,Fornecedor 02,18,9,50.00
36,2025-01,Fornecedor 02,14,7,50.00
32,2024-11,Fornecedor 09,12,6,50.00


### 5.11 — Desvio mensal em relação à taxa geral do fornecedor


In [ ]:
taxa_geral_fornecedor = (
    df_atrasos_mensais
    .groupby("nome_fornecedor")
    .agg(
        pedidos_total=("pedido_id", "nunique"),
        pedidos_atrasados_total=("pedido_atrasado", "sum")
    )
    .reset_index()
)

taxa_geral_fornecedor["taxa_atraso_geral"] = (
    taxa_geral_fornecedor["pedidos_atrasados_total"]
    / taxa_geral_fornecedor["pedidos_total"] * 100
)

comparacao_mensal = resumo_mensal.merge(
    taxa_geral_fornecedor[
        ["nome_fornecedor", "taxa_atraso_geral"]
    ],
    on="nome_fornecedor",
    how="left"
)

comparacao_mensal["desvio_pontos_percentuais"] = (
    comparacao_mensal["percentual_atrasados"]
    - comparacao_mensal["taxa_atraso_geral"]
)

display(
    comparacao_mensal
    .sort_values("desvio_pontos_percentuais", ascending=False)
    .head(10)[
        [
            "mes",
            "nome_fornecedor",
            "quantidade_pedidos",
            "pedidos_atrasados",
            "percentual_atrasados",
            "taxa_atraso_geral",
            "desvio_pontos_percentuais"
        ]
    ].round(2)
)


,mes,nome_fornecedor,quantidade_pedidos,pedidos_atrasados,percentual_atrasados,taxa_atraso_geral,desvio_pontos_percentuais
41,2025-02,Fornecedor 09,15,9,60.00,36.45,23.55
29,2024-10,Fornecedor 09,12,7,58.33,36.45,21.88
27,2024-10,Fornecedor 02,16,9,56.25,35.32,20.93
48,2025-05,Fornecedor 02,16,9,56.25,35.32,20.93
16,2024-06,Fornecedor 05,18,10,55.56,34.90,20.65
33,2024-12,Fornecedor 02,9,5,55.56,35.32,20.23
7,2024-03,Fornecedor 05,11,6,54.55,34.90,19.64
36,2025-01,Fornecedor 02,14,7,50.00,35.32,14.68
15,2024-06,Fornecedor 02,18,9,50.00,35.32,14.68
35,2024-12,Fornecedor 09,12,6,50.00,36.45,13.55


### 5.12 — Resumo executivo dos fornecedores críticos


In [ ]:
resumo_executivo_fornecedores = (
    df_diagnostico
    .merge(
        resumo_atrasos,
        on="nome_fornecedor",
        how="left"
    )
    .sort_values(
        ["percentual_otif", "gasto_total"],
        ascending=[True, False]
    )
)

display(
    resumo_executivo_fornecedores[
        [
            "nome_fornecedor",
            "gasto_total",
            "quantidade_pedidos",
            "percentual_otif",
            "percentual_atrasados",
            "pedidos_atrasados",
            "atraso_medio_dias",
            "atraso_maximo_dias"
        ]
    ].round(2)
)

,nome_fornecedor,gasto_total,quantidade_pedidos,percentual_otif,percentual_atrasados,pedidos_atrasados,atraso_medio_dias,atraso_maximo_dias
2,Fornecedor 09,7161608.40,299,58.86,36.45,109,5.76,33.0
0,Fornecedor 02,8134214.60,402,59.45,35.32,142,5.20,35.0
1,Fornecedor 05,10521630.42,361,59.83,34.90,126,5.43,24.0


## 6. Análise de oportunidades de economia

A estimativa abaixo é uma oportunidade teórica calculada com base nos preços observados. Ela deve ser validada considerando especificações, período, fornecedor, volume, frete e condições comerciais antes de ser tratada como oportunidade negociável.

### 6.1 — Ranking dos produtos por economia potencial

### 6.1 — Ranking dos produtos por economia potencial


In [ ]:

df_top_economia = (
    df_oportunidades
    .sort_values(
        "economia_potencial",
        ascending=False
    )
    .head(10)
)

display(
    df_top_economia[
        [
            "produto_id",
            "nome_produto",
            "quantidade_total",
            "gasto_total",
            "preco_medio",
            "menor_preco",
            "economia_potencial"
        ]
    ].round(2)
)

,produto_id,nome_produto,quantidade_total,gasto_total,preco_medio,menor_preco,economia_potencial
87,178,Produto 178,13900,7476785.69,533.23,233.59,4165039.39
30,126,Produto 126,17603,6344395.72,360.11,190.09,2992870.06
284,86,Produto 086,16883,5487712.46,325.69,163.99,2730018.01
213,291,Produto 291,13677,4173884.83,306.94,137.18,2321812.38
159,242,Produto 242,10704,4368701.41,404.22,195.46,2234615.88
249,54,Produto 054,26734,4443137.52,165.79,84.63,2169618.33
239,45,Produto 045,23193,3660808.33,156.86,69.64,2022945.81
192,272,Produto 272,9043,3893212.62,426.66,219.42,1874063.73
170,252,Produto 252,5887,3310324.50,562.79,245.23,1869459.97
162,245,Produto 245,5322,3601450.30,654.83,341.23,1669002.25


### 6.2 — Top 10 produtos por economia potencial

In [ ]:

df_top_economia = (
    df_oportunidades
    .nlargest(10, "economia_potencial")
    .sort_values("economia_potencial", ascending=True)
)

fig_economia = px.bar(
    df_top_economia,
    x="economia_potencial",
    y="nome_produto",
    orientation="h",
    title="Top 10 Produtos por Economia Potencial",
    labels={
        "economia_potencial": "Economia potencial (R$)",
        "nome_produto": "Produto"
    },
    hover_data={
        "quantidade_total": True,
        "gasto_total": ":,.2f",
        "preco_medio": ":,.2f",
        "menor_preco": ":,.2f"
    }
)

fig_economia.update_layout(
    template="plotly_white",
    title_x=0.5,
    separators=",.",
    height=550,
    xaxis_title="Economia potencial (R$)",
    yaxis_title="Produto"
)

fig_economia.update_xaxes(
    tickprefix="R$ ",
    tickformat=",.0f"
)

fig_economia.show()


### 6.3 — Concentração das oportunidades de economia

In [ ]:
economia_total = df_oportunidades["economia_potencial"].sum()

economia_top10 = df_top_economia["economia_potencial"].sum()

participacao_top10_economia = (
    economia_top10 / economia_total * 100
    if economia_total > 0 else 0
)

print(f"Economia potencial total: R$ {economia_total:,.2f}")
print(f"Economia potencial do Top 10: R$ {economia_top10:,.2f}")
print(f"Participação do Top 10: {participacao_top10_economia:.2f}%")
print(f"Demais produtos: R$ {economia_total - economia_top10:,.2f}")

Economia potencial total: R$ 62,807,385.70
Economia potencial do Top 10: R$ 24,049,445.81
Participação do Top 10: 38.29%
Demais produtos: R$ 38,757,939.89


### 6.4 — Distribuição da economia potencial

In [ ]:
df_concentracao_economia = pd.DataFrame({
    "Grupo": ["Top 10 produtos", "Demais produtos"],
    "Economia potencial": [
        economia_top10,
        economia_total - economia_top10
    ]
})

fig_concentracao_economia = px.pie(
    df_concentracao_economia,
    names="Grupo",
    values="Economia potencial",
    hole=0.55,
    title="Distribuição da Economia Potencial",
    color="Grupo",
    color_discrete_map={
        "Top 10 produtos": "#636EFA",
        "Demais produtos": "#B8C2CC"
    }
)

fig_concentracao_economia.update_traces(
    textposition="inside",
    texttemplate="%{percent:.1%}",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Valor: R$ %{value:,.2f}<br>"
        "Participação: %{percent:.2%}"
        "<extra></extra>"
    )
)

fig_concentracao_economia.update_layout(
    template="plotly_white",
    title_x=0.5,
    separators=",."
)

fig_concentracao_economia.show()

## 7. Análise de risco de abastecimento

### 7.1 — Classificação do risco de abastecimento


In [ ]:
df_risco = pd.read_sql_query("""
    SELECT *
    FROM vw_risco_abastecimento
""", conn)

resumo_risco = (
    df_risco
    .groupby("classificacao_risco")
    .agg(
        quantidade_produtos=("produto_id", "nunique"),
        gasto_total=("gasto_total", "sum")
    )
    .reset_index()
    .sort_values("gasto_total", ascending=False)
)

display(resumo_risco.round(2))

print(f"Produtos analisados: {df_risco['produto_id'].nunique():,}")
print(f"Gasto total analisado: R$ {df_risco['gasto_total'].sum():,.2f}")


,classificacao_risco,quantidade_produtos,gasto_total
1,BAIXO RISCO,275,1.363102e+08
3,RISCO MODERADO,7,5.703024e+06
0,ALTO RISCO,1,6.054582e+05
2,DEPENDÊNCIA DE FORNECEDOR,16,3.610717e+05


Produtos analisados: 299
Gasto total analisado: R$ 142,979,742.46


### 7.2 — Produtos com maior risco de abastecimento

A tabela lista os produtos fora da classificação de baixo risco. A categoria é um sinal para investigação, não uma confirmação de risco operacional.

#### Tabela detalhada dos produtos fora de baixo risco


In [ ]:
df_produtos_risco = (
    df_risco[
        df_risco["classificacao_risco"] != "BAIXO RISCO"
    ]
    .sort_values("gasto_total", ascending=False)
)

display(
    df_produtos_risco[
        [
            "produto_id",
            "nome_produto",
            "quantidade_fornecedores",
            "quantidade_pedidos",
            "gasto_total",
            "classificacao_risco"
        ]
    ].round(2)
)

,produto_id,nome_produto,quantidade_fornecedores,quantidade_pedidos,gasto_total,classificacao_risco
4,102,Produto 102,2,86,1172026.97,RISCO MODERADO
120,207,Produto 207,2,58,998559.16,RISCO MODERADO
46,140,Produto 140,2,30,861032.64,RISCO MODERADO
191,271,Produto 271,2,70,841550.97,RISCO MODERADO
153,237,Produto 237,2,41,719897.17,RISCO MODERADO
121,208,Produto 208,1,29,605458.24,ALTO RISCO
198,278,Produto 278,2,26,603322.22,RISCO MODERADO
243,49,Produto 049,2,60,506634.49,RISCO MODERADO
2,100,Produto 100,1,33,244695.00,DEPENDÊNCIA DE FORNECEDOR
29,125,Produto 125,1,14,27671.57,DEPENDÊNCIA DE FORNECEDOR


### 7.3 — Gastos por classificação de risco

In [ ]:
fig_gastos_risco = px.bar(
    resumo_risco.sort_values("gasto_total", ascending=True),
    x="gasto_total",
    y="classificacao_risco",
    orientation="h",
    text="quantidade_produtos",
    title="Gasto Total por Classificação de Risco",
    labels={
        "gasto_total": "Gasto associado (R$)",
        "classificacao_risco": "Classificação de risco",
        "quantidade_produtos": "Quantidade de produtos"
    },
    color="classificacao_risco",
    color_discrete_map={
        "BAIXO RISCO": "#16A34A",
        "RISCO MODERADO": "#E5A23B",
        "ALTO RISCO": "#DC6262",
        "DEPENDÊNCIA DE FORNECEDOR": "#6478F8"
    }
)

fig_gastos_risco.update_layout(
    template="plotly_white",
    title_x=0.5,
    separators=",.",
    height=450,
    showlegend=False
)

fig_gastos_risco.update_xaxes(
    tickprefix="R$ ",
    tickformat=",.0f"
)

fig_gastos_risco.update_traces(
    texttemplate="%{text} produtos",
    textposition="outside",
    cliponaxis=False
)

fig_gastos_risco.show()


### 7.4 — Priorização de produtos com risco de abastecimento

In [ ]:
df_produtos_risco["percentual_gasto_empresa"] = (
    df_produtos_risco["gasto_total"]
    / gasto_empresa * 100
)

display(
    df_produtos_risco[
        [
            "produto_id",
            "nome_produto",
            "quantidade_fornecedores",
            "quantidade_pedidos",
            "gasto_total",
            "percentual_gasto_empresa",
            "classificacao_risco"
        ]
    ]
    .sort_values("gasto_total", ascending=False)
    .round(2)
)

print(
    f"Produtos fora da classificação de baixo risco: "
    f"{len(df_produtos_risco)}"
)

print(
    f"Gasto associado: "
    f"R$ {df_produtos_risco['gasto_total'].sum():,.2f}"
)

print(
    f"Participação no gasto total: "
    f"{df_produtos_risco['gasto_total'].sum() / gasto_empresa * 100:.2f}%"
)


,produto_id,nome_produto,quantidade_fornecedores,quantidade_pedidos,gasto_total,percentual_gasto_empresa,classificacao_risco
4,102,Produto 102,2,86,1172026.97,0.82,RISCO MODERADO
120,207,Produto 207,2,58,998559.16,0.70,RISCO MODERADO
46,140,Produto 140,2,30,861032.64,0.60,RISCO MODERADO
191,271,Produto 271,2,70,841550.97,0.59,RISCO MODERADO
153,237,Produto 237,2,41,719897.17,0.50,RISCO MODERADO
121,208,Produto 208,1,29,605458.24,0.42,ALTO RISCO
198,278,Produto 278,2,26,603322.22,0.42,RISCO MODERADO
243,49,Produto 049,2,60,506634.49,0.35,RISCO MODERADO
2,100,Produto 100,1,33,244695.00,0.17,DEPENDÊNCIA DE FORNECEDOR
29,125,Produto 125,1,14,27671.57,0.02,DEPENDÊNCIA DE FORNECEDOR


Produtos fora da classificação de baixo risco: 24
Gasto associado: R$ 6,669,553.51
Participação no gasto total: 4.66%


### 7.5 — Recomendações para mitigação do risco de abastecimento

As recomendações abaixo decorrem dos indicadores disponíveis. A criticidade dos produtos, o lead time e a existência de alternativas qualificadas devem ser verificados antes de definir ações definitivas.

Recomendações

1. Priorizar produtos com alto gasto e poucas alternativas

O Produto 102 apresenta o maior gasto entre os produtos classificados fora de baixo risco, com R\$ 1,17 milhão e dois fornecedores. Recomenda-se avaliar a capacidade de fornecimento dos parceiros atuais e buscar alternativas qualificadas.

2. Investigar dependência de fornecedor único

O Produto 208 foi classificado como alto risco e depende de um único fornecedor. Recomenda-se verificar a criticidade operacional, o prazo de reposição e a viabilidade de homologar um fornecedor alternativo.

3. Avaliar a relevância operacional dos produtos de menor gasto

Produtos com gasto reduzido também podem representar risco caso sejam críticos para a operação ou não possuam substitutos. A priorização deve considerar critérios operacionais além do valor comprado.

4. Aprimorar o modelo de classificação

A classificação atual utiliza quantidade de fornecedores e um limite de gasto definido para a simulação. Uma versão futura pode incorporar criticidade, lead time, histórico de atrasos e disponibilidade de alternativas para produzir uma avaliação mais completa.

## 8. Consolidação e conclusão executiva

Esta tabela reúne os principais KPIs e resultados das análises anteriores para facilitar a leitura executiva.

### 8.1 — Consolidação dos indicadores executivos


In [ ]:
gasto_total_projeto = gasto_empresa
quantidade_pedidos_validos = len(df_validos)
otif_global = df_validos["otif"].mean() * 100
percentual_pedidos_atrasados = (
    (df_validos["dias_atraso"] > 0).mean() * 100
)
ticket_medio = df_validos["valor_pedido"].mean()

# Desempenho dos fornecedores prioritários
gasto_fornecedores_prioritarios = df_prioridades["gasto_total"].sum()
participacao_fornecedores_prioritarios = (
    gasto_fornecedores_prioritarios / gasto_total_projeto * 100
)

# Oportunidades de economia
economia_potencial_total = df_oportunidades["economia_potencial"].sum()
economia_potencial_top10 = df_top_economia["economia_potencial"].sum()

# Risco de abastecimento
quantidade_produtos_risco = len(df_produtos_risco)
gasto_produtos_risco = df_produtos_risco["gasto_total"].sum()
participacao_produtos_risco = (
    gasto_produtos_risco / gasto_total_projeto * 100
)

indicadores_executivos = pd.DataFrame({
    "Indicador": [
        "Gasto total",
        "Pedidos válidos",
        "OTIF global",
        "Pedidos atrasados",
        "Ticket médio",
        "Gasto associado aos fornecedores prioritários",
        "Participação dos fornecedores prioritários",
        "Economia potencial teórica",
        "Economia potencial teórica do Top 10",
        "Produtos fora da classificação de baixo risco",
        "Gasto associado aos produtos fora de baixo risco",
        "Participação desses produtos no gasto total"
    ],
    "Resultado": [
        f"R$ {gasto_total_projeto:,.2f}",
        f"{quantidade_pedidos_validos:,}",
        f"{otif_global:.2f}%",
        f"{percentual_pedidos_atrasados:.2f}%",
        f"R$ {ticket_medio:,.2f}",
        f"R$ {gasto_fornecedores_prioritarios:,.2f}",
        f"{participacao_fornecedores_prioritarios:.2f}%",
        f"R$ {economia_potencial_total:,.2f}",
        f"R$ {economia_potencial_top10:,.2f}",
        f"{quantidade_produtos_risco}",
        f"R$ {gasto_produtos_risco:,.2f}",
        f"{participacao_produtos_risco:.2f}%"
    ]
})

display(indicadores_executivos)

,Indicador,Resultado
0,Gasto total,"R$ 142,979,742.46"
1,Pedidos válidos,"7,772"
2,OTIF global,71.69%
3,Pedidos atrasados,23.01%
4,Ticket médio,"R$ 18,396.78"
5,Gasto associado aos fornecedores prioritários,"R$ 67,535,677.09"
6,Participação dos fornecedores prioritários,47.23%
7,Economia potencial teórica,"R$ 62,807,385.70"
8,Economia potencial teórica do Top 10,"R$ 24,049,445.81"
9,Produtos fora da classificação de baixo risco,24


### 8.2 — Conclusão executiva e recomendações

O projeto Procurement Analytics analisou dados de compras da empresa fictícia Nexa Distribuição Ltda., abrangendo 8.000 pedidos registrados entre janeiro de 2024 e dezembro de 2025, além de fornecedores, produtos, categorias e entregas. Após excluir os pedidos cancelados das análises pertinentes, foram considerados 7.772 pedidos válidos, associados a um gasto total de R\$ 142,98 milhões.

1. Desempenho de fornecedores

O OTIF global foi de 71,69%, enquanto 23,01% dos pedidos apresentaram atraso. A análise identificou 12 fornecedores prioritários, selecionados por apresentarem gasto acima da mediana e OTIF abaixo da mediana. Esse grupo representa 47,23% dos gastos da empresa.

Entre os casos investigados, os Fornecedores 02, 05 e 09 apresentaram OTIF inferior a 60%. O Fornecedor 09 registrou o menor OTIF e o maior atraso médio entre os três; o Fornecedor 02 apresentou o maior número de pedidos atrasados; e o Fornecedor 05 concentrou o maior gasto. Esses resultados sustentam a recomendação de investigar causas de atraso, revisar níveis de serviço e desenvolver planos de melhoria com os parceiros prioritários.

2. Oportunidades de economia

A análise de preços identificou uma economia potencial teórica de R\$ 62,81 milhões. Esse valor foi calculado a partir da diferença entre o preço médio observado e o menor preço observado para cada produto.

Os dez produtos com maior potencial concentram R\$ 24,05 milhões, equivalentes a 38,29% da estimativa total.

Esses valores devem ser tratados como oportunidades para investigação, e não como economia garantida. Antes de negociar, é necessário validar a comparabilidade dos preços, considerando período, fornecedor, volume, especificações, frete e condições comerciais.

3. Risco de abastecimento

A classificação de risco identificou 24 produtos fora da categoria de baixo risco, associados a R\$ 6,67 milhões em gastos, ou 4,66% do total. O Produto 208 foi classificado como alto risco por depender de um único fornecedor, enquanto o Produto 102 apresentou o maior gasto entre os produtos fora da categoria de baixo risco.

Recomenda-se avaliar a criticidade operacional desses produtos, os prazos de reposição e a possibilidade de homologar fornecedores alternativos. A classificação atual é uma triagem inicial baseada em quantidade de fornecedores e limites de gasto definidos para a simulação.

4. Recomendações prioritárias

Investigar atrasos dos fornecedores prioritários, começando pelos Fornecedores 02, 05 e 09.

Validar comercialmente as oportunidades de preço dos produtos com maior potencial teórico.

Avaliar alternativas de fornecimento para produtos dependentes de um único fornecedor.

Monitorar periodicamente gasto, OTIF, pontualidade, completude e concentração de fornecedores.

Evoluir a classificação de risco com dados de criticidade, lead time e disponibilidade de alternativas.

Considerações finais

O projeto demonstra como SQL, Python, Pandas e Plotly podem ser combinados para transformar dados transacionais em indicadores e análises aplicáveis à gestão de compras. Os resultados oferecem uma base para priorizar investigações, apoiar negociações e melhorar o acompanhamento de fornecedores, respeitando as limitações dos dados e das estimativas produzidas